# EAIM-Net v5 — Complete Pipeline with Step-by-Step Visualization

Run cells **top to bottom**. Each section shows progress with charts, images, and metric tables.

| Cell | Step | Visualization |
|------|------|---------------|
| 1 | Setup | Checkpoint list, dataset counts, device info |
| 2 | Dataset Diagnosis | Pair counts per dataset, sample image grid |
| 3 | Pre-train LowLight | Live loss/PSNR curve + before/after grid |
| 4 | Pre-train Dehazing | Live loss/PSNR curve + before/after grid |
| 5 | Pre-train Rain | Live loss/PSNR curve + before/after grid |
| 6 | Pre-train Glare | Live loss/PSNR curve + glare map overlay |
| 7 | Pre-train IllumNorm | Live loss/PSNR curve + before/after grid |
| 8 | Assemble Model | Filter weight summary, EPE health check |
| 9 | Joint Fine-tuning | Live training dashboard: loss, H, τ, W-acc |
| 10 | Single Image Test | Deep-dive: 3 figures, metrics card |
| 11 | Full Evaluation | PSNR/SSIM/LPIPS table, heatmap, gallery |
| 12 | TensorBoard | Training curves |


---
## Cell 1 — Setup

In [ ]:
# ================================================================
# CELL 1 - SETUP  (run once per session)
# ================================================================
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
pip("lpips","scikit-image","ipywidgets","tqdm","pyyaml","matplotlib")

import os, glob, json, math, time, random, warnings, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as TF
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.notebook import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
import lpips as lpips_lib
import ipywidgets as WG
from IPython.display import display as _D, clear_output
warnings.filterwarnings("ignore")

# ================================================================
# PATHS  -  ONLY EDIT THESE TWO LINES
# ================================================================
COLAB_ROOT  = "/content/eaim_v5"
WEATHER_DIR = "/content/drive/MyDrive/Weather"
#   ^^ Upload LOL.zip, RESIDE_SOTS.zip, Rain100L.zip, Rain100H.zip,
#      DID-MDN.zip, SD1.zip, weather_time_data.zip to this folder.
#      Everything else is automatic.

DRIVE_ROOT    = "/content/drive/MyDrive/adaptive_enhancement_training"
# ================================================================

DRIVE_CKPTS   = f"{DRIVE_ROOT}/checkpoints_v5"
DRIVE_PRE     = f"{DRIVE_ROOT}/pretrained_filters"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results_v5"
DRIVE_LOGS    = f"{DRIVE_ROOT}/logs_v5"
DATA_LOCAL    = "/content/weather_time_data"
LOG_FILE      = f"{DRIVE_ROOT}/training_log_v5.txt"

# ── Step 1: Create ALL directories first ─────────────────────
os.makedirs(COLAB_ROOT, exist_ok=True)          # MUST be before os.chdir
for d in [DRIVE_CKPTS, DRIVE_PRE, DRIVE_RESULTS, DRIVE_LOGS]:
    os.makedirs(d, exist_ok=True)

# ── Step 2: Change to working directory ──────────────────────
os.chdir(COLAB_ROOT)
sys.path.insert(0, COLAB_ROOT)

# ── Step 3: Copy source .py files from Drive → COLAB_ROOT ────
# Upload all .py files from src_v5/ folder on Drive once.
# They auto-copy here every session.
DRIVE_SRC = f"{DRIVE_ROOT}/src_v5"
import shutil as _shutil

PY_FILES = [
    "afb_module.py", "ess_module.py", "epe_module.py", "complete_model.py",
    "config.py", "dataset.py", "dataset_real_pairs.py",
    "losses.py", "dataset_loader.py", "dataset_loader_lazy.py",
    "pretrain_filters.py", "assemble_and_finetune.py",
]

print(f"Copying source files from {DRIVE_SRC} ...")
_missing = []
for _f in PY_FILES:
    _src = os.path.join(DRIVE_SRC, _f)
    _dst = os.path.join(COLAB_ROOT, _f)
    if os.path.exists(_src):
        _shutil.copy2(_src, _dst)
        print(f"  OK    {_f}")
    elif os.path.exists(_dst):
        print(f"  CACHE {_f}  (already present)")
    else:
        _missing.append(_f)
        print(f"  MISS  {_f}")

if _missing:
    print(f"\nWARNING: {len(_missing)} files missing from {DRIVE_SRC}")
    print("Please upload the src_v5/ folder to your Drive and re-run Cell 1.")
else:
    print(f"\nAll {len(PY_FILES)} source files ready in {COLAB_ROOT}")

# ── Auto-extract all datasets from zip files ──────────────────
from dataset_loader import (load_all_datasets, load_all_datasets_multi_drive,
                             mount_second_drive, load_sd1_from_second_drive,
                             get_loader_kwargs, PATHS)

# ================================================================
# SMART LAZY LOADING - datasets extracted ONE AT A TIME
# ================================================================
# No datasets are extracted here at setup time.
# Each pre-training cell (3-7) extracts only what IT needs,
# then cleans up before the next cell runs.
#
# This means:
#   - LOL.zip    (~0.3 GB) extracted only during Cell 3
#   - RESIDE.zip (~0.8 GB) extracted only during Cell 4
#   - Rain zips  (~1.5 GB) extracted only during Cell 5
#   - SD1.zip    (~10 GB)  extracted only during Cell 6, then deleted
#   - WTT.zip    (~0.2 GB) extracted only during Cell 7
#
# Maximum disk usage at any time: ~10 GB (only during Cell 6)
# After Cell 6 finishes: SD1 is deleted, disk drops back to ~0 GB
#
# SD1 on a second Google account? See Cell 6 for the commented
# mount_second_drive() option - no Drive space needed on either account.
# ================================================================

# Just verify the Weather folder exists and list available zips
if os.path.isdir(WEATHER_DIR):
    zips = sorted([f for f in os.listdir(WEATHER_DIR) if f.lower().endswith(".zip")])
    print(f"Weather folder: {WEATHER_DIR}")
    print(f"Zips available ({len(zips)}):")
    for z in zips:
        mb = os.path.getsize(os.path.join(WEATHER_DIR, z))/1e6
        print(f"  {z:<35} {mb:>8.1f} MB")
else:
    print(f"WARNING: Weather folder not found: {WEATHER_DIR}")
    print("Create it on Drive and upload your zip files.")

# These will be set by each pre-training cell when it runs
LOL_ROOT = RESIDE_ROOT = RAIN100L_ROOT = RAIN100H_ROOT = DIDMDN_ROOT = SD1_ROOT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FILTER_NAMES  = ["Low-light","Dehazing","Rain Removal","Illum. Norm.","Glare Reduction"]
WEATHER_NAMES = ["Clear","Rain","Fog","Light Snow","Glare"]
TIME_NAMES    = ["Dawn","Day","Dusk","Night"]
ILLUM_NAMES   = ["Low","Medium","High"]
FILTER_COLORS = ["#2278CF","#1D9E75","#EF9F27","#7F77DD","#D85A30"]

# ── Shared losses ─────────────────────────────────────────────
def _gauss(s=11,sig=1.5):
    c=torch.arange(s,dtype=torch.float32)-s//2
    g=torch.exp(-c**2/(2*sig**2)); return (g/g.sum()).outer(g/g.sum())

def ssim_loss(p,g,ws=11):
    C1,C2=0.01**2,0.03**2; B,C,H,W=p.shape
    win=_gauss(ws).to(p.device).expand(C,1,ws,ws); pad=ws//2
    mu1=TF.conv2d(p,win,padding=pad,groups=C); mu2=TF.conv2d(g,win,padding=pad,groups=C)
    s1=TF.conv2d(p*p,win,padding=pad,groups=C)-mu1**2
    s2=TF.conv2d(g*g,win,padding=pad,groups=C)-mu2**2
    s12=TF.conv2d(p*g,win,padding=pad,groups=C)-mu1*mu2
    return 1-(((2*mu1*mu2+C1)*(2*s12+C2))/((mu1**2+mu2**2+C1)*(s1+s2+C2))).mean()

def base_loss(pred,gt,ls=0.3): return nn.L1Loss()(pred,gt)+ls*ssim_loss(pred,gt)

def text_mask(gt,ks=5,th=0.02):
    gray=0.299*gt[:,0:1]+0.587*gt[:,1:2]+0.114*gt[:,2:3]
    lk=torch.tensor([[0.,-1.,0.],[-1.,4.,-1.],[0.,-1.,0.]],device=gt.device).view(1,1,3,3)
    return (TF.max_pool2d(TF.conv2d(gray,lk,padding=1).abs(),ks,1,ks//2)>th).float()

def text_aware_loss(pred,gt):
    return base_loss(pred,gt)+2.0*((pred-gt).abs()*text_mask(gt).detach()).mean()

# ── Shared metrics ────────────────────────────────────────────
LPIPS_FN = lpips_lib.LPIPS(net="alex").to(DEVICE)

def metrics(pred_np, gt_np):
    p=psnr_fn(gt_np,pred_np,data_range=1.0)
    s=ssim_fn(gt_np,pred_np,data_range=1.0,channel_axis=2)
    pt=torch.from_numpy(pred_np).permute(2,0,1).unsqueeze(0).to(DEVICE)
    gt=torch.from_numpy(gt_np ).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): lp=LPIPS_FN(pt*2-1,gt*2-1).item()
    return {"psnr":p,"ssim":s,"lpips":lp}

# ── Checkpoint helpers ────────────────────────────────────────
def _valid(p):
    if not os.path.exists(p) or os.path.getsize(p)<1024: return False
    try:
        import zipfile
        with zipfile.ZipFile(p,"r") as z: return len(z.namelist())>0
    except: return False

def find_latest(d):
    for p in sorted(glob.glob(os.path.join(d,"checkpoint_epoch_*.pth")),reverse=True):
        if _valid(p): return p
    best=os.path.join(d,"best_model.pth")
    return best if _valid(best) else None

def _exists(p): return bool(p) and os.path.isdir(p) and len(os.listdir(p))>0

def cleanup_old(d,keep=5):
    for p in sorted(glob.glob(os.path.join(d,"checkpoint_epoch_*.pth")))[:-keep]:
        try: os.remove(p)
        except: pass

def save_filter(m,name):
    p=os.path.join(DRIVE_PRE,f"{name}.pth")
    torch.save(m.state_dict(),p); print(f"  saved: {p}")

def load_filter(m,name):
    p=os.path.join(DRIVE_PRE,f"{name}.pth")
    if os.path.exists(p):
        m.load_state_dict(torch.load(p,map_location=DEVICE)); print(f"  loaded: {p}"); return True
    return False

# ── Visualization helpers ─────────────────────────────────────
def show_image_grid(images, titles, ncols=4, figsize_per=3.2, suptitle=""):
    nrows=math.ceil(len(images)/ncols)
    fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*figsize_per,nrows*figsize_per))
    axes=np.array(axes).reshape(-1)
    for i,(img,title) in enumerate(zip(images,titles)):
        axes[i].imshow(img); axes[i].axis("off"); axes[i].set_title(title,fontsize=8)
    for j in range(i+1,len(axes)): axes[j].axis("off")
    if suptitle: fig.suptitle(suptitle,fontsize=11,fontweight="bold")
    plt.tight_layout(); plt.show()

def before_after_grid(module, dataset, n=6, title=""):
    module.eval()
    idxs=random.sample(range(len(dataset)),min(n,len(dataset)))
    imgs,titles=[],[]
    with torch.no_grad():
        for i in idxs:
            batch=dataset[i]
            inp=batch["input"].unsqueeze(0).to(DEVICE)
            tgt=batch["target"].unsqueeze(0).to(DEVICE)
            st=torch.ones(1,1).to(DEVICE)
            if "glare_map" in batch:
                pred=torch.clamp(module(inp,st,batch["glare_map"].unsqueeze(0).to(DEVICE)),0,1)
            else:
                pred=torch.clamp(module(inp,st),0,1)
            inp_np=inp[0].cpu().numpy().transpose(1,2,0)
            tgt_np=tgt[0].cpu().numpy().transpose(1,2,0)
            pred_np=pred[0].cpu().numpy().transpose(1,2,0)
            psnr_in=psnr_fn(tgt_np,inp_np,data_range=1.0)
            psnr_out=psnr_fn(tgt_np,pred_np,data_range=1.0)
            imgs+=[np.clip(inp_np,0,1),np.clip(pred_np,0,1),np.clip(tgt_np,0,1)]
            titles+=[f"Input\n{psnr_in:.1f}dB",f"Enhanced\n{psnr_out:.1f}dB","GT"]
    show_image_grid(imgs,titles,ncols=6,suptitle=title)

# ── Dataset helpers ───────────────────────────────────────────
def _to_sq(pil,sz): return pil.resize((sz,sz),Image.LANCZOS)

def _aug(inp,gt,sz):
    w,h=inp.size
    if w>sz and h>sz:
        x=random.randint(0,w-sz); y=random.randint(0,h-sz)
        inp=inp.crop((x,y,x+sz,y+sz)); gt=gt.crop((x,y,x+sz,y+sz))
    else: inp=_to_sq(inp,sz); gt=_to_sq(gt,sz)
    if random.random()>0.5:
        inp=inp.transpose(Image.FLIP_LEFT_RIGHT); gt=gt.transpose(Image.FLIP_LEFT_RIGHT)
    return inp,gt

TT=transforms.ToTensor()

def _find(root,inp_names,tgt_names,split=None):
    bases=[root]
    if split: bases=[os.path.join(root,split)]+bases
    for base in bases:
        if not os.path.isdir(base): continue
        subs={d.lower():d for d in os.listdir(base) if os.path.isdir(os.path.join(base,d))}
        i=next((os.path.join(base,subs[k]) for k in inp_names if k in subs),None)
        t=next((os.path.join(base,subs[k]) for k in tgt_names if k in subs),None)
        if i and t: return i,t
    raise FileNotFoundError(f"dirs not found under {root}")

# ── Dataset classes ───────────────────────────────────────────
class PairDS(Dataset):
    def __init__(self,inp_dir,tgt_dir,sz=256,augment=False,label=""):
        self.pairs=[]; self.sz=sz; self.aug=augment
        exts=(".jpg",".jpeg",".png")
        for f in sorted(os.listdir(inp_dir)):
            if not f.lower().endswith(exts): continue
            stem=os.path.splitext(f)[0]
            gt_stem=stem.split("_")[0] if "_" in stem else stem
            gt=None
            for gs in [stem,gt_stem]:
                for ext in exts:
                    c=os.path.join(tgt_dir,gs+ext)
                    if os.path.exists(c): gt=c; break
                if gt: break
            if gt: self.pairs.append((os.path.join(inp_dir,f),gt))
        print(f"  {label}: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,gp=self.pairs[idx]
        inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
        if self.aug: inp,gt=_aug(inp,gt,self.sz)
        else: inp=_to_sq(inp,self.sz); gt=_to_sq(gt,self.sz)
        return {"input":TT(inp),"target":TT(gt),"name":os.path.basename(ip)}

class SD1DS(Dataset):
    def __init__(self,root,sz=256,augment=False):
        self.samples=[]; self.sz=sz; self.aug=augment
        for f in sorted(os.listdir(root)):
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                self.samples.append(os.path.join(root,f))
        print(f"  SD1 strip: {len(self.samples)} images")
    def __len__(self): return len(self.samples)
    def __getitem__(self,idx):
        img=Image.open(self.samples[idx]).convert("RGB"); w,h=img.size; pw=w//3
        clean=_to_sq(img.crop((0,0,pw,h)),self.sz)
        glare=_to_sq(img.crop((pw,0,pw*2,h)),self.sz)
        gmap =_to_sq(img.crop((pw*2,0,w,h)),self.sz)
        if self.aug and random.random()>0.5:
            clean=clean.transpose(Image.FLIP_LEFT_RIGHT)
            glare=glare.transpose(Image.FLIP_LEFT_RIGHT)
            gmap =gmap.transpose(Image.FLIP_LEFT_RIGHT)
        return {"input":TT(glare),"target":TT(clean),"glare_map":TT(gmap.convert("L")),"name":os.path.basename(self.samples[idx])}

class SD1TestDS(Dataset):
    def __init__(self,root,sz=256):
        self.pairs=[]; self.sz=sz; exts=(".jpg",".jpeg",".png")
        gt_d=os.path.join(root,"gt"); lt_d=os.path.join(root,"light")
        if os.path.isdir(gt_d) and os.path.isdir(lt_d):
            for f in sorted(os.listdir(gt_d)):
                if f.lower().endswith(exts):
                    lf=os.path.join(lt_d,f)
                    if os.path.exists(lf): self.pairs.append((lf,os.path.join(gt_d,f)))
        else:
            for gf in sorted(glob.glob(os.path.join(root,"*_gt.*"))):
                stem=os.path.basename(gf).split("_gt")[0]
                for ext in exts:
                    lf=os.path.join(root,stem+"_light"+ext)
                    if os.path.exists(lf): self.pairs.append((lf,gf)); break
        print(f"  SD1 test: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        lp,gp=self.pairs[idx]
        inp=Image.open(lp).convert("RGB").resize((self.sz,self.sz),Image.LANCZOS)
        gt =Image.open(gp).convert("RGB").resize((self.sz,self.sz),Image.LANCZOS)
        return {"input":TT(inp),"target":TT(gt),"name":os.path.basename(lp)}

class WTTDS(Dataset):
    def __init__(self,root,split="train",sz=256):
        self.pairs=[]; self.sz=sz; self.aug=(split=="train")
        sd=os.path.join(root,split)
        id_=os.path.join(sd,"images"); cd=os.path.join(sd,"clean_images")
        if not os.path.isdir(id_): return
        for f in sorted(os.listdir(id_)):
            if not f.lower().endswith((".jpg",".jpeg",".png")): continue
            cp=os.path.join(cd,f)
            if os.path.exists(cp): self.pairs.append((os.path.join(id_,f),cp))
        print(f"  WTT {split}: {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,gp=self.pairs[idx]
        inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
        if self.aug: inp,gt=_aug(inp,gt,self.sz)
        else: inp=_to_sq(inp,self.sz); gt=_to_sq(gt,self.sz)
        return {"input":TT(inp),"target":TT(gt)}

# ── Filter trainer with live visualization ─────────────────────
def train_filter_viz(module,tr_ld,va_ld,name,n_epochs=25,lr=2e-4,loss_fn=None,color="#2278CF"):
    module=module.to(DEVICE)

    # ── Resume: load best weights + read log to find completed epochs ────
    best_psnr=0.0
    start_ep=0
    log_path=os.path.join(DRIVE_PRE,f"{name}_log.txt")

    if load_filter(module,f"{name}_best"):
        # Count how many epochs already done from log file
        if os.path.exists(log_path):
            with open(log_path) as _lf:
                completed=[l for l in _lf.readlines() if l.strip().startswith("Ep")]
                start_ep=len(completed)
                # Read best PSNR from log
                psnr_vals=[float(l.split("psnr=")[1].split("dB")[0])
                           for l in completed if "psnr=" in l]
                if psnr_vals: best_psnr=max(psnr_vals)
        if start_ep>=n_epochs:
            print(f"  [{name}] Already completed {start_ep}/{n_epochs} epochs.")
            print(f"  Best PSNR={best_psnr:.4f} dB. Skipping training.")
            return module
        print(f"  [{name}] Resuming from epoch {start_ep}/{n_epochs}  Best PSNR so far={best_psnr:.4f}")

    remaining=n_epochs-start_ep
    opt=optim.Adam(module.parameters(),lr=lr,weight_decay=1e-5)
    sch=optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=1,eta_min=1e-6)
    # Fast-forward scheduler to match current epoch
    for _fep in range(start_ep): sch.step(_fep+1)

    ep_list=[]; loss_list=[]; psnr_list=[]; lines=[]
    out_curve=WG.Output(); out_status=WG.Output()
    _D(WG.VBox([WG.HTML(f"<h3 style='color:#1B3A6B'>Pre-training: {name} (ep {start_ep}→{n_epochs})</h3>"),out_status,out_curve]))
    for ep in range(start_ep, n_epochs):
        module.train(); tot=0.0
        for batch in tqdm(tr_ld,desc=f"{name} ep{ep:02d}",leave=False):
            inp=batch["input"].to(DEVICE); tgt=batch["target"].to(DEVICE)
            st=torch.ones(inp.shape[0],1).to(DEVICE)
            pred=module(inp,st,batch["glare_map"].to(DEVICE)) if name=="glare" and "glare_map" in batch else module(inp,st)
            loss=(loss_fn(pred,tgt,batch) if loss_fn else base_loss(pred,tgt))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(module.parameters(),1.0)
            opt.step(); tot+=loss.item()
        sch.step(ep+1)
        module.eval(); psnrs,ssims=[],[]
        with torch.no_grad():
            for batch in va_ld:
                inp=batch["input"].to(DEVICE); tgt=batch["target"].to(DEVICE)
                st=torch.ones(inp.shape[0],1).to(DEVICE)
                pred=torch.clamp(module(inp,st),0,1)
                for i in range(inp.shape[0]):
                    pn=np.clip(pred[i].cpu().numpy().transpose(1,2,0),0,1)
                    gn=np.clip(tgt[i].cpu().numpy().transpose(1,2,0),0,1)
                    psnrs.append(psnr_fn(gn,pn,data_range=1.0))
                    ssims.append(ssim_fn(gn,pn,data_range=1.0,channel_axis=2))
        vp=float(np.mean(psnrs)); vs=float(np.mean(ssims))
        lr_now=opt.param_groups[0]["lr"]; avg=tot/len(tr_ld)
        ep_list.append(ep); loss_list.append(avg); psnr_list.append(vp)
        line=f"Ep{ep:02d}  loss={avg:.4f}  psnr={vp:.3f}dB  ssim={vs:.4f}  lr={lr_now:.2e}"
        lines.append(line)
        with open(log_path,"a") as _lf: _lf.write(line+"\n")
        if vp > best_psnr + 0.005:
            best_psnr=vp
            save_filter(module, f"{name}_best")
        with out_status:
            clear_output(wait=True)
            print(f"  Epoch {ep+1}/{n_epochs}  Loss={avg:.4f}  PSNR={vp:.3f}dB  SSIM={vs:.4f}")
            print(f"  Best PSNR: {best_psnr:.4f} dB")
        if len(ep_list)>=2 and (ep%2==0 or ep==n_epochs-1):
            with out_curve:
                clear_output(wait=True)
                fig,(a1,a2)=plt.subplots(1,2,figsize=(12,3.5))
                a1.plot(ep_list,loss_list,color=color,lw=2,marker="o",ms=4)
                a1.set_title("Training Loss"); a1.set_xlabel("Epoch"); a1.grid(True,alpha=0.3)
                a2.plot(ep_list,psnr_list,color="#E8541A",lw=2,marker="s",ms=4)
                a2.set_title("Val PSNR (dB)"); a2.set_xlabel("Epoch"); a2.grid(True,alpha=0.3)
                fig.suptitle(f"{name} Pre-training",fontsize=11,fontweight="bold")
                plt.tight_layout(); plt.show()
    # Log already written per-epoch above
    print(f"\n  DONE [{name}]  Best PSNR={best_psnr:.4f} dB")
    return module

# ── Checkpoint info ───────────────────────────────────────────
gpu=torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU"
print(f"Device : {DEVICE}  {gpu}")
print(f"\ncheckpoints_v5:")
for p in sorted(glob.glob(os.path.join(DRIVE_CKPTS,"*.pth"))):
    print(f"  {os.path.basename(p):<45} {os.path.getsize(p)/1e6:>6.1f} MB  {'OK' if _valid(p) else 'CORRUPT'}")
print("\nSetup complete. Datasets extracted above. Run Cell 2 next.")


---
## Cell 0b — Quick Resume After Disconnect
*Run Cell 1 then this cell to restore all variables after a session disconnect. Then jump directly to your target cell.*

In [ ]:
# ================================================================
# CELL 0b — QUICK RESUME AFTER DISCONNECT
# Run this instead of Cells 2-7 when reconnecting mid-training.
# It restores all variables needed to continue from where you stopped.
#
# USAGE:
#   1. Run Cell 1 (Setup) — always required after disconnect
#   2. Run THIS cell — restores paths and variables
#   3. Jump directly to whichever cell you were on (3b/4/4b/5/6/7)
# ================================================================

print("Restoring session variables after disconnect...")

# ── Restore dataset paths from already-extracted folders ─────────
from dataset_loader_lazy import LOCAL_BASE, _EXTRACTED
import os

# Check what is already extracted on local disk
_datasets = {
    "LOL":        "/content/datasets/LOL",
    "RESIDE_SOTS":"/content/datasets/RESIDE_SOTS",
    "Rain100L":   "/content/datasets/Rain100L",
    "Rain100H":   "/content/datasets/Rain100H",
    "DID-MDN":    "/content/datasets/DID-MDN",
    "SD1":        "/content/datasets/SD1",
    "WTT":        "/content/datasets/WTT",
}

print("\nLocal disk status:")
for name, path in _datasets.items():
    if os.path.isdir(path):
        n = sum(1 for _,_,fs in os.walk(path) for f in fs
                if f.lower().endswith((".jpg",".png",".jpeg")))
        print(f"  OK    {name:<14} {n:>6} images  →  {path}")
    else:
        print(f"  MISS  {name:<14} (not on local disk — will re-extract if needed)")

# ── Restore convenience variables ─────────────────────────────────
from dataset_loader_lazy import _resolve_lol, _resolve_reside, _resolve_rain, _resolve_didmdn

def _get(path, resolver=None):
    if not os.path.isdir(path): return None
    return resolver(path) if resolver else path

LOL_ROOT      = _get("/content/datasets/LOL",        _resolve_lol)
RESIDE_ROOT   = _get("/content/datasets/RESIDE_SOTS", _resolve_reside)
RAIN100L_ROOT = _get("/content/datasets/Rain100L",   _resolve_rain)
RAIN100H_ROOT = _get("/content/datasets/Rain100H",   _resolve_rain)
DIDMDN_ROOT   = _get("/content/datasets/DID-MDN",    _resolve_didmdn)
SD1_ROOT      = _get("/content/datasets/SD1")
DATA_LOCAL    = "/content/datasets/WTT" if os.path.isdir("/content/datasets/WTT") else DATA_LOCAL

print("\nRestored path variables:")
for name, val in [("LOL_ROOT",LOL_ROOT),("RESIDE_ROOT",RESIDE_ROOT),
                   ("RAIN100L_ROOT",RAIN100L_ROOT),("RAIN100H_ROOT",RAIN100H_ROOT),
                   ("DIDMDN_ROOT",DIDMDN_ROOT),("SD1_ROOT",SD1_ROOT)]:
    status = "OK" if val and os.path.isdir(val) else "MISS"
    print(f"  {status}  {name} = {val}")

# ── Show pretrained filters status ────────────────────────────────
print("\nPretrained filters saved on Drive:")
import glob
pths = sorted(glob.glob(os.path.join(DRIVE_PRE, "*_best.pth")))
if pths:
    for p in pths:
        mb = os.path.getsize(p)/1e6
        print(f"  {os.path.basename(p):<35} {mb:>6.1f} MB")
else:
    print("  (none yet)")

# ── Glare-specific: show where training left off ───────────────────
glare_best = os.path.join(DRIVE_PRE, "glare_best.pth")
if os.path.exists(glare_best):
    import torch
    state = torch.load(glare_best, map_location="cpu")
    n_params = sum(v.numel() for v in state.values())
    print(f"\nglare_best.pth  →  {n_params:,} params loaded")
    print("Cell 6 will resume from this checkpoint automatically.")
    print("→ Re-run Cell 6 now (it calls load_filter at the top)")
else:
    print("\nNo glare_best.pth found — Cell 6 will start from scratch.")

print("\nResume complete. Jump to your target cell now.")
print()
print("IMPORTANT — After reconnect, datasets are NOT on local disk.")
print("Each cell auto-extracts what it needs from WEATHER_DIR zips.")
print()
print("For fine-tuning (Cell 9):")
print("  Cell 9 will auto-extract LOL + WTT + Rain100L before training.")
print("  Just run Cell 1 → Cell 0b → Cell 9.")
print()
print("For any pre-training cell (3-7):")
print("  Each cell extracts its own zip automatically.")
print("  Just run Cell 1 → Cell 0b → target cell.")


In [ ]:
# ================================================================
# CELL 2 — DATASET VERIFICATION AND SAMPLE IMAGES
# Auto-loads from zips. Shows what was found and sample images.
# ================================================================
from dataset_loader import PATHS, print_paths

# Show current paths
print_paths()

# ── Sample images from each dataset ──────────────────────────
print("\nLoading sample images for visual verification...")
samples=[]; sample_titles=[]

# LOL
if PATHS["lol"]:
    try:
        inp_d,tgt_d=_find(PATHS["lol"],["low"],["high"],"our485")
        files=sorted([f for f in os.listdir(inp_d) if f.lower().endswith((".jpg",".png"))])[:2]
        for fname in files:
            ip=Image.open(os.path.join(inp_d,fname)).convert("RGB")
            for ext in [".png",".jpg",".jpeg"]:
                gp_=os.path.join(tgt_d,os.path.splitext(fname)[0]+ext)
                if os.path.exists(gp_): break
            gp=Image.open(gp_).convert("RGB") if os.path.exists(gp_) else ip
            samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                      np.array(_to_sq(gp,224)).astype(np.float32)/255]
            sample_titles+=[f"LOL\nInput",f"LOL\nGT (high/)"]
    except Exception as e: print(f"  LOL sample: {e}")

# RESIDE
if PATHS["reside"]:
    try:
        hd=os.path.join(PATHS["reside"],"hazy")
        cd=os.path.join(PATHS["reside"],"clear")
        if not os.path.isdir(hd): hd=os.path.join(PATHS["reside"],"indoor","hazy"); cd=os.path.join(PATHS["reside"],"indoor","clear")
        files=sorted([f for f in os.listdir(hd) if f.lower().endswith((".jpg",".png"))])[:2]
        for fname in files:
            stem=os.path.splitext(fname)[0].split("_")[0]
            ip=Image.open(os.path.join(hd,fname)).convert("RGB")
            gp_=None
            for ext in [".png",".jpg"]:
                c=os.path.join(cd,stem+ext)
                if os.path.exists(c): gp_=c; break
            gp=Image.open(gp_).convert("RGB") if gp_ else ip
            samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                      np.array(_to_sq(gp,224)).astype(np.float32)/255]
            sample_titles+=[f"RESIDE\nHazy",f"RESIDE\nClear"]
    except Exception as e: print(f"  RESIDE sample: {e}")

# Rain100L
if PATHS["rain100l"]:
    try:
        ri,rt=_find(PATHS["rain100l"],["rain"],["norain","clean"],"train")
        files=sorted([f for f in os.listdir(ri) if f.lower().endswith((".jpg",".png"))])[:2]
        for fname in files:
            ip=Image.open(os.path.join(ri,fname)).convert("RGB")
            for ext in [".png",".jpg"]:
                gp_=os.path.join(rt,os.path.splitext(fname)[0]+ext)
                if os.path.exists(gp_): break
            gp=Image.open(gp_).convert("RGB") if os.path.exists(gp_) else ip
            samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                      np.array(_to_sq(gp,224)).astype(np.float32)/255]
            sample_titles+=[f"Rain100L\nRain",f"Rain100L\nClean"]
    except Exception as e: print(f"  Rain100L sample: {e}")

# DID-MDN
if PATHS["didmdn"]:
    try:
        ri=os.path.join(PATHS["didmdn"],"train","rain")
        rt=os.path.join(PATHS["didmdn"],"train","clear")
        if os.path.isdir(ri):
            files=sorted([f for f in os.listdir(ri) if f.lower().endswith((".jpg",".png"))])[:2]
            for fname in files:
                ip=Image.open(os.path.join(ri,fname)).convert("RGB")
                for ext in [".png",".jpg"]:
                    gp_=os.path.join(rt,os.path.splitext(fname)[0]+ext)
                    if os.path.exists(gp_): break
                gp=Image.open(gp_).convert("RGB") if os.path.exists(gp_) else ip
                samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                          np.array(_to_sq(gp,224)).astype(np.float32)/255]
                sample_titles+=[f"DID-MDN\nRain",f"DID-MDN\nClear"]
    except Exception as e: print(f"  DID-MDN sample: {e}")

# SD1
if PATHS["sd1"]:
    try:
        td=None
        for sub in ["train","Train",""]:
            p=os.path.join(PATHS["sd1"],sub) if sub else PATHS["sd1"]
            if os.path.isdir(p):
                files=[f for f in os.listdir(p) if f.lower().endswith((".jpg",".png",".bmp"))]
                if files: td=p; break
        if td:
            img=Image.open(os.path.join(td,sorted(files)[0])).convert("RGB")
            w,h=img.size; pw=w//3
            clean=np.array(_to_sq(img.crop((0,0,pw,h)),224)).astype(np.float32)/255
            glare=np.array(_to_sq(img.crop((pw,0,pw*2,h)),224)).astype(np.float32)/255
            gmap_pil=_to_sq(img.crop((pw*2,0,w,h)),224).convert("L")
            gmap=np.array(gmap_pil).astype(np.float32)/255
            samples+=[clean,glare,np.stack([gmap,gmap,gmap],axis=2)]
            sample_titles+=["SD1\nGT","SD1\nGlare","SD1\nMap"]
    except Exception as e: print(f"  SD1 sample: {e}")

# WTT
if PATHS["wtt"]:
    try:
        id_=os.path.join(PATHS["wtt"],"train","images")
        cd=os.path.join(PATHS["wtt"],"train","clean_images")
        if os.path.isdir(id_):
            files=sorted([f for f in os.listdir(id_) if f.lower().endswith((".jpg",".png"))])[:2]
            for fname in files:
                ip=Image.open(os.path.join(id_,fname)).convert("RGB")
                gp_=os.path.join(cd,fname)
                gp=Image.open(gp_).convert("RGB") if os.path.exists(gp_) else ip
                samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                          np.array(_to_sq(gp,224)).astype(np.float32)/255]
                sample_titles+=[f"WTT\nDegraded",f"WTT\nClean GT"]
    except Exception as e: print(f"  WTT sample: {e}")

if samples:
    show_image_grid(samples, sample_titles, ncols=4,
                    suptitle="Dataset Sample Images — Left=Degraded  Right=Clean GT")
else:
    print("No sample images to show. Check that datasets extracted correctly.")


---
## Cell 2 — Dataset Diagnosis

In [ ]:
# ================================================================
# CELL 2 — DATASET DIAGNOSIS
# Shows: pair counts, sample images from each dataset
# ================================================================
from dataset import diagnose_lol
from dataset_real_pairs import debug_dataset_roots

debug_dataset_roots(
    synthetic_root = DATA_LOCAL    if _exists(DATA_LOCAL)    else None,
    lol_root       = LOL_ROOT      if _exists(LOL_ROOT)      else None,
    rain100l_root  = RAIN100L_ROOT if _exists(RAIN100L_ROOT) else None,
    rain100h_root  = RAIN100H_ROOT if _exists(RAIN100H_ROOT) else None,
)
if _exists(LOL_ROOT): diagnose_lol(LOL_ROOT)

# ── Sample images from each dataset ──────────────────────────
print("\nLoading sample images for visual check...")
samples=[]; sample_titles=[]

datasets_to_show=[
    (LOL_ROOT,    "our485","low","high",  "LOL (low → high)"),
    (RAIN100L_ROOT,"train","rain","norain","Rain100L (rain → clean)"),
    (RESIDE_ROOT,  None,   "hazy","clear","RESIDE (hazy → clear)"),
]
for root,split,inp_n,tgt_n,label in datasets_to_show:
    if not root or not os.path.isdir(root): continue
    try:
        inp_d,tgt_d=_find(root,[inp_n],[tgt_n],split)
        files=sorted([f for f in os.listdir(inp_d) if f.lower().endswith((".jpg",".png"))])
        for fname in files[:2]:
            ip=Image.open(os.path.join(inp_d,fname)).convert("RGB")
            gp=Image.open(os.path.join(tgt_d,os.path.splitext(fname)[0]+".png") if
                          os.path.exists(os.path.join(tgt_d,os.path.splitext(fname)[0]+".png"))
                          else os.path.join(tgt_d,fname)).convert("RGB")
            samples+=[np.array(_to_sq(ip,224)).astype(np.float32)/255,
                      np.array(_to_sq(gp,224)).astype(np.float32)/255]
            sample_titles+=[f"{label}\nInput",f"{label}\nClean GT"]
    except Exception as e: print(f"  {label} skipped: {e}")

# SD1
if _exists(SD1_ROOT):
    try:
        td=None
        for sub in ["train","Train"]:
            p=os.path.join(SD1_ROOT,sub)
            if os.path.isdir(p): td=p; break
        if td is None: td=SD1_ROOT
        files=sorted([f for f in os.listdir(td) if f.lower().endswith((".jpg",".png"))])[:1]
        for fname in files:
            img=Image.open(os.path.join(td,fname)).convert("RGB"); w,h=img.size; pw=w//3
            clean=np.array(_to_sq(img.crop((0,0,pw,h)),224)).astype(np.float32)/255
            glare=np.array(_to_sq(img.crop((pw,0,pw*2,h)),224)).astype(np.float32)/255
            gmap =np.array(_to_sq(img.crop((pw*2,0,w,h)),224).convert("L")).astype(np.float32)/255
            samples+=[clean,glare,np.stack([gmap,gmap,gmap],axis=2)]
            sample_titles+=["SD1 GT","SD1 Glare","SD1 Map"]
    except Exception as e: print(f"  SD1 skipped: {e}")

if samples:
    show_image_grid(samples,sample_titles,ncols=4,suptitle="Dataset Sample Images (Input | GT)")
else:
    print("No sample images found. Check dataset paths.")


---
## Cell 3 — Pre-train LowLight on LOL

In [ ]:
# ================================================================
# CELL 3 — PRE-TRAIN LowLight filter on LOL
# Shows: live loss/PSNR curves, before/after grid at end
# ================================================================

# ── Load LOL dataset (extract from zip on demand) ────────────
from dataset_loader_lazy import load_for_filter, cleanup_after_filter, disk_usage
_paths = load_for_filter("lowlight", WEATHER_DIR)
LOL_ROOT = _paths.get("lol")
if LOL_ROOT is None:
    print("LOL.zip not found in", WEATHER_DIR, "- skipping LowLight pre-training")
else:

from afb_module import LowLightEnhancementModule
EPOCHS_LL=25; BSZ_LL=8; SZ_LL=256

def _find_lol_split(root, split):
    """
    Robust LOL split finder. Handles all common layouts:
      root/our485/low + root/our485/high     (standard train)
      root/eval15/low + root/eval15/high     (standard val)
      root/train/low  + root/train/high      (alternate)
      root/low        + root/high            (flat)
      nested after extraction at any depth
    """
    if split in ("train","tr"):
        sub_priority = ["our485","train"]
    else:
        sub_priority = ["eval15","test","val"]

    # Walk up to 4 levels deep
    for r, dirs, _ in os.walk(root):
        if r[len(root):].count(os.sep) > 4: continue
        dc = {d.lower():d for d in dirs}
        # Check preferred subdirs first
        for sub in sub_priority:
            if sub in dc:
                candidate = os.path.join(r, dc[sub])
                cd = {d.lower():d for d in os.listdir(candidate)
                      if os.path.isdir(os.path.join(candidate,d))}
                if "low" in cd and "high" in cd:
                    inp = os.path.join(candidate, cd["low"])
                    tgt = os.path.join(candidate, cd["high"])
                    n_i = len([f for f in os.listdir(inp) if f.lower().endswith((".jpg",".png",".jpeg"))])
                    n_t = len([f for f in os.listdir(tgt) if f.lower().endswith((".jpg",".png",".jpeg"))])
                    if n_i > 0 and n_t > 0:
                        return inp, tgt
        # Also check if r itself contains low/ and high/
        if "low" in dc and "high" in dc:
            inp = os.path.join(r, dc["low"])
            tgt = os.path.join(r, dc["high"])
            n_i = len([f for f in os.listdir(inp) if f.lower().endswith((".jpg",".png",".jpeg"))])
            if n_i > 0: return inp, tgt
    raise FileNotFoundError(f"LOL {split} dirs not found under {root}. Tree:\n" +
                            "\n".join(f"  {r}" for r,_,_ in list(os.walk(root))[:20]))

try:
    tr_i,tr_t = _find_lol_split(LOL_ROOT, "train")
    va_i,va_t = _find_lol_split(LOL_ROOT, "val")
    print(f"  train input : {tr_i}")
    print(f"  train target: {tr_t}")
    print(f"  val   input : {va_i}")
    print(f"  val   target: {va_t}")
    tr_ds=PairDS(tr_i,tr_t,SZ_LL,augment=True, label="LOL-train")
    va_ds=PairDS(va_i,va_t,SZ_LL,augment=False,label="LOL-val")
    tr_ld=DataLoader(tr_ds,batch_size=BSZ_LL,shuffle=True, num_workers=2,pin_memory=True)
    va_ld=DataLoader(va_ds,batch_size=BSZ_LL,shuffle=False,num_workers=2,pin_memory=True)
    ll_mod=LowLightEnhancementModule()
    ll_mod=train_filter_viz(ll_mod,tr_ld,va_ld,"lowlight",
                            n_epochs=EPOCHS_LL,lr=2e-4,color="#2278CF")
    print("\nBefore/After on validation set:")
    before_after_grid(ll_mod,va_ds,n=6,title="LowLight: Input | Enhanced | GT")
except Exception as e:
    import traceback
    print(f"LOL pre-training error: {e}")
    traceback.print_exc()

# ── Cleanup: free local disk + remove periodic Drive checkpoints ──────
# Uncomment cleanup_after_filter to free local disk before Cell 4 runs:
# cleanup_after_filter("lowlight")   # frees LOL (~0.3 GB)

# Remove periodic ep*** checkpoints from Drive (keep only _best.pth)
import glob as _gl
pre_dir = DRIVE_PRE
ep_ckpts = sorted(_gl.glob(os.path.join(pre_dir, "lowlight_ep*.pth")))
if ep_ckpts:
    print(f"Removing {len(ep_ckpts)} periodic checkpoints from Drive...")
    for p in ep_ckpts:
        os.remove(p)
        print(f"  removed: {os.path.basename(p)}")
    print("Done. Only lowlight_best.pth kept.")
else:
    print("No periodic checkpoints to remove.")

best = os.path.join(pre_dir, "lowlight_best.pth")
if os.path.exists(best):
    sz = os.path.getsize(best)/1e6
    print(f"lowlight_best.pth: {sz:.1f} MB  ← this is all we need")


---
## Cell 3b — Quick Test: LowLight filter on one image
*Run after Cell 3 to verify the filter before moving to Dehazing.*

In [ ]:
# ================================================================
# CELL 3b — QUICK TEST: LowLight filter on one image
# Run this right after Cell 3 to verify the filter works.
#
# HOW TO USE:
#   Option A: Leave TEST_IMAGE = None  → picks a random LOL eval15 image
#   Option B: Set TEST_IMAGE = "/path/to/your/dark_image.jpg"
# ================================================================

TEST_IMAGE = None   # ← set your own image path here, or leave None

import os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
from torchvision import transforms

from afb_module import LowLightEnhancementModule

# ── Load best weights ─────────────────────────────────────────
best_path = os.path.join(DRIVE_PRE, "lowlight_best.pth")
assert os.path.exists(best_path), f"Not found: {best_path} — run Cell 3 first"

ll_mod = LowLightEnhancementModule().to(DEVICE)
ll_mod.load_state_dict(torch.load(best_path, map_location=DEVICE))
ll_mod.eval()
print(f"Loaded: {best_path}")

# ── Pick test image ───────────────────────────────────────────
gt_path  = None
inp_path = TEST_IMAGE

if inp_path is None:
    # Auto-pick from LOL eval15
    try:
        va_i, va_t = _find_lol_split(LOL_ROOT, "val")
        files = sorted([f for f in os.listdir(va_i)
                        if f.lower().endswith((".jpg",".png",".jpeg"))])
        fname    = random.choice(files)
        inp_path = os.path.join(va_i, fname)
        # Find GT
        stem = os.path.splitext(fname)[0]
        for ext in [".png",".jpg",".jpeg"]:
            c = os.path.join(va_t, stem+ext)
            if os.path.exists(c): gt_path=c; break
        print(f"Auto-selected: {fname}")
    except Exception as e:
        print(f"Could not auto-select from LOL: {e}")
        print("Set TEST_IMAGE manually and re-run.")
        raise

assert inp_path and os.path.exists(inp_path), f"Image not found: {inp_path}"

# ── Load and enhance ──────────────────────────────────────────
def load_np(p): return np.array(Image.open(p).convert("RGB")).astype(np.float32)/255

inp_np = load_np(inp_path)
gt_np  = load_np(gt_path) if gt_path else None

# Resize to model-friendly size (keep aspect ratio)
inp_pil  = Image.open(inp_path).convert("RGB")
IW, IH  = inp_pil.size
# Pad to multiple of 4
PW = ((IW+3)//4)*4; PH = ((IH+3)//4)*4
inp_pad  = Image.new("RGB",(PW,PH)); inp_pad.paste(inp_pil,(0,0))

inp_t = transforms.ToTensor()(inp_pad).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    for strength_val in [0.5, 0.75, 1.0]:
        st = torch.ones(1,1).to(DEVICE) * strength_val
        out = torch.clamp(ll_mod(inp_t, st), 0, 1)
        if strength_val == 1.0:
            enh_np = out[0].cpu().numpy().transpose(1,2,0)[:IH,:IW]

# Also get intermediate strengths for comparison strip
enh_outputs = {}
with torch.no_grad():
    for sv in [0.25, 0.5, 0.75, 1.0]:
        st = torch.ones(1,1).to(DEVICE) * sv
        o  = torch.clamp(ll_mod(inp_t, st), 0, 1)
        enh_outputs[sv] = o[0].cpu().numpy().transpose(1,2,0)[:IH,:IW]

# ── Metrics ───────────────────────────────────────────────────
has_gt = gt_np is not None
if has_gt:
    # Resize gt to match inp if needed
    if gt_np.shape != inp_np.shape:
        gt_pil2 = Image.open(gt_path).convert("RGB").resize((IW,IH), Image.LANCZOS)
        gt_np   = np.array(gt_pil2).astype(np.float32)/255
    m_inp = {"psnr": psnr_fn(gt_np,inp_np,data_range=1.0),
             "ssim": ssim_fn(gt_np,inp_np,data_range=1.0,channel_axis=2)}
    m_enh = {"psnr": psnr_fn(gt_np,enh_np,data_range=1.0),
             "ssim": ssim_fn(gt_np,enh_np,data_range=1.0,channel_axis=2)}
    dpsnr = m_enh["psnr"] - m_inp["psnr"]
    dssim = m_enh["ssim"] - m_inp["ssim"]

# ── Figure 1: Main comparison ─────────────────────────────────
ncols = 3 if has_gt else 2
fig1, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5.5))

axes[0].imshow(np.clip(inp_np,0,1))
axes[0].axis("off")
t0 = f"Input (Degraded)"
if has_gt: t0 += f"\nPSNR={m_inp['psnr']:.2f} dB   SSIM={m_inp['ssim']:.4f}"
axes[0].set_title(t0, fontsize=10)

ok   = has_gt and dpsnr >= 0
col  = "#1a7a1a" if ok else ("#cc2222" if has_gt else "black")
axes[1].imshow(np.clip(enh_np,0,1))
axes[1].axis("off")
t1 = "Enhanced (LowLight filter, strength=1.0)"
if has_gt:
    t1 += f"\nPSNR={m_enh['psnr']:.2f} dB   SSIM={m_enh['ssim']:.4f}"
    t1 += f"\nΔPSNR={dpsnr:+.2f}   ΔSSIM={dssim:+.4f}"
axes[1].set_title(t1, fontsize=10, color=col)

if has_gt:
    axes[2].imshow(np.clip(gt_np,0,1))
    axes[2].axis("off")
    axes[2].set_title("Ground Truth (LOL high/)", fontsize=10)

fig1.suptitle(f"LowLight Filter Test — {os.path.basename(inp_path)}",
              fontsize=12, fontweight="bold")
plt.tight_layout()
fig1_path = os.path.join(DRIVE_RESULTS, "lowlight_test_comparison.jpg")
fig1.savefig(fig1_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {fig1_path}")

# ── Figure 2: Strength sweep (0.25 → 0.50 → 0.75 → 1.00) ────
fig2, axes2 = plt.subplots(1, 4, figsize=(22, 5))
for idx, (sv, enh_out) in enumerate(sorted(enh_outputs.items())):
    axes2[idx].imshow(np.clip(enh_out,0,1))
    axes2[idx].axis("off")
    t = f"Strength = {sv:.2f}"
    if has_gt:
        p = psnr_fn(gt_np, enh_out, data_range=1.0)
        s = ssim_fn(gt_np, enh_out, data_range=1.0, channel_axis=2)
        t += f"\nPSNR={p:.2f}   SSIM={s:.4f}"
    axes2[idx].set_title(t, fontsize=9)
fig2.suptitle("Strength Sweep — how much enhancement to apply",
              fontsize=11, fontweight="bold")
plt.tight_layout()
fig2_path = os.path.join(DRIVE_RESULTS, "lowlight_strength_sweep.jpg")
fig2.savefig(fig2_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {fig2_path}")

# ── Figure 3: Difference map ──────────────────────────────────
diff     = np.abs(enh_np - inp_np)           # how much changed
diff_amp = np.clip(diff * 5, 0, 1)            # amplified x5 to see subtle changes

fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5))
axes3[0].imshow(np.clip(inp_np,0,1));  axes3[0].axis("off"); axes3[0].set_title("Input", fontsize=10)
axes3[1].imshow(np.clip(enh_np,0,1)); axes3[1].axis("off"); axes3[1].set_title("Enhanced", fontsize=10)
im = axes3[2].imshow(diff_amp, cmap="hot"); axes3[2].axis("off")
axes3[2].set_title("Difference Map (×5)\nBright = changed a lot\nDark = unchanged", fontsize=9)
plt.colorbar(im, ax=axes3[2], shrink=0.8)
fig3.suptitle("What the filter changed — LowLight",
              fontsize=11, fontweight="bold")
plt.tight_layout()
fig3_path = os.path.join(DRIVE_RESULTS, "lowlight_diff_map.jpg")
fig3.savefig(fig3_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {fig3_path}")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*50)
print("  LowLight Filter Test Result")
print("="*50)
print(f"  Image  : {os.path.basename(inp_path)}")
print(f"  Size   : {IW}×{IH}")
if has_gt:
    print(f"  PSNR   : {m_inp['psnr']:.2f} dB  →  {m_enh['psnr']:.2f} dB  ({dpsnr:+.2f})")
    print(f"  SSIM   : {m_inp['ssim']:.4f}   →  {m_enh['ssim']:.4f}   ({dssim:+.4f})")
    print(f"  Status : {'✓ IMPROVED' if dpsnr>0 else '✗ REGRESSED'}")
else:
    print("  No GT provided — visual check only")
print("="*50)
print("\nIf the enhanced image looks good → proceed to Cell 4 (Dehazing)")
print("If not → check that lowlight_best.pth loaded correctly above.")


---
## Cell 4 — Pre-train Dehazing on RESIDE_SOTS

In [ ]:
# ================================================================
# CELL 4 — PRE-TRAIN Dehazing filter on RESIDE_SOTS
# Shows: live loss/PSNR curves, before/after grid at end
# ================================================================

# ── Load RESIDE dataset (extract from zip on demand) ─────────
from dataset_loader_lazy import load_for_filter, cleanup_after_filter, disk_usage
_paths = load_for_filter("dehazing", WEATHER_DIR)
RESIDE_ROOT = _paths.get("reside")
if RESIDE_ROOT is None:
    print("RESIDE_SOTS.zip not found in", WEATHER_DIR, "- skipping Dehazing pre-training")
else:

from afb_module import DehazingModule
EPOCHS_DH=25; BSZ_DH=8; SZ_DH=256

try:
    # ── RESIDE-ITS: hazy/1000_10_0.74905.png → clear/1000.png ─────
    # Format: stem before FIRST underscore = GT image ID
    # 1399 clear images × 10 hazy variants = 13,990 pairs total

    class ResideITSDataset(Dataset):
        def __init__(self, hazy_dir, clear_dir, size=256, augment=False):
            self.pairs=[]; self.size=size; self.aug=augment
            exts=(".jpg",".jpeg",".png")
            # Build clear lookup: "1000" -> full path
            clear_lut={}
            for f in os.listdir(clear_dir):
                if f.lower().endswith(exts):
                    clear_lut[os.path.splitext(f)[0]]=os.path.join(clear_dir,f)
            # Pair each hazy image with its GT
            for f in sorted(os.listdir(hazy_dir)):
                if not f.lower().endswith(exts): continue
                gt_stem=os.path.splitext(f)[0].split("_")[0]  # "1000_10_0.74" -> "1000"
                if gt_stem in clear_lut:
                    self.pairs.append((os.path.join(hazy_dir,f), clear_lut[gt_stem]))
            print(f"  RESIDE-ITS: {len(self.pairs)} pairs "
                  f"({len(clear_lut)} clear imgs x ~10 hazy each)")

        def __len__(self): return len(self.pairs)

        def __getitem__(self,idx):
            ip,gp=self.pairs[idx]
            inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
            if self.aug:
                iw,ih=inp.size
                if iw>self.size and ih>self.size:
                    x=random.randint(0,iw-self.size); y=random.randint(0,ih-self.size)
                    inp=inp.crop((x,y,x+self.size,y+self.size))
                    gt =gt.crop( (x,y,x+self.size,y+self.size))
                else:
                    inp=inp.resize((self.size,self.size),Image.LANCZOS)
                    gt =gt.resize( (self.size,self.size),Image.LANCZOS)
                if random.random()>0.5:
                    inp=inp.transpose(Image.FLIP_LEFT_RIGHT)
                    gt =gt.transpose(Image.FLIP_LEFT_RIGHT)
            else:
                inp=inp.resize((self.size,self.size),Image.LANCZOS)
                gt =gt.resize( (self.size,self.size),Image.LANCZOS)
            to_t=transforms.ToTensor()
            return {"input":to_t(inp),"target":to_t(gt)}

    # Find hazy/ and clear/ inside RESIDE_ROOT (handles any nesting depth)
    def _find_reside_dirs(root):
        for r,dirs,_ in os.walk(root):
            if r[len(root):].count(os.sep)>3: continue
            dc={d.lower():d for d in dirs}
            if "hazy" in dc and "clear" in dc:
                return os.path.join(r,dc["hazy"]),os.path.join(r,dc["clear"])
        raise FileNotFoundError(f"hazy/ and clear/ not found under {root}")

    hazy_d,clear_d=_find_reside_dirs(RESIDE_ROOT)
    print(f"  hazy : {hazy_d}")
    print(f"  clear: {clear_d}")

    full_ds=ResideITSDataset(hazy_d,clear_d,SZ_DH,augment=True)
    n_va=max(1,len(full_ds)//5); n_tr=len(full_ds)-n_va
    tr_ds,va_ds=torch.utils.data.random_split(
        full_ds,[n_tr,n_va],generator=torch.Generator().manual_seed(42))
    tr_ld=DataLoader(tr_ds,batch_size=BSZ_DH,shuffle=True, num_workers=2,pin_memory=True)
    va_ld=DataLoader(va_ds,batch_size=BSZ_DH,shuffle=False,num_workers=2,pin_memory=True)
    dh_mod=DehazingModule()
    dh_mod=train_filter_viz(dh_mod,tr_ld,va_ld,"dehazing",
                            n_epochs=EPOCHS_DH,lr=2e-4,color="#1D9E75")
    print("\nBefore/After on validation set:")
    before_after_grid(dh_mod,va_ds.dataset,n=6,title="Dehazing: Input | Enhanced | GT")
except Exception as e:
    print(f"Dehazing pre-training skipped: {e}")

# Optional: free disk before next cell
# cleanup_after_filter("dehazing")   # frees RESIDE (~0.8 GB) before Rain loads


---
## Cell 4b — Quick Test: Dehazing filter
*Run after Cell 4 before moving to Rain Removal.*

In [ ]:
# ================================================================
# CELL 4b — QUICK TEST: Dehazing filter on one image
# Auto-picks a hazy/clear pair from RESIDE val split.
# Set TEST_HAZY / TEST_CLEAR to use your own images.
# ================================================================
TEST_HAZY  = None   # e.g. "/content/my_foggy.jpg"
TEST_CLEAR = None   # ground truth, or None for no metrics

import os, random
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
from afb_module import DehazingModule

best_path = os.path.join(DRIVE_PRE, "dehazing_best.pth")
assert os.path.exists(best_path), f"Not found: {best_path} — run Cell 4 first"
dh_mod = DehazingModule().to(DEVICE)
dh_mod.load_state_dict(torch.load(best_path, map_location=DEVICE))
dh_mod.eval()
print(f"Loaded: {best_path}")

# Auto-pick from RESIDE val
if TEST_HAZY is None:
    try:
        hazy_d2, clear_d2 = _find_reside_dirs(RESIDE_ROOT)
        hazy_files = sorted([f for f in os.listdir(hazy_d2)
                             if f.lower().endswith((".jpg",".png",".jpeg"))])
        fname   = random.choice(hazy_files)
        gt_stem = os.path.splitext(fname)[0].split("_")[0]
        TEST_HAZY = os.path.join(hazy_d2, fname)
        for ext in [".png",".jpg",".jpeg"]:
            c = os.path.join(clear_d2, gt_stem+ext)
            if os.path.exists(c): TEST_CLEAR=c; break
        print(f"Auto-selected: {fname}  →  GT: {gt_stem}")
    except Exception as e:
        print(f"Auto-select failed: {e}. Set TEST_HAZY manually.")
        raise

def _load(p): return np.array(Image.open(p).convert("RGB")).astype(np.float32)/255
def _enh(pil_img, mod):
    iw,ih = pil_img.size
    pw,ph = ((iw+3)//4)*4, ((ih+3)//4)*4
    pad   = Image.new("RGB",(pw,ph)); pad.paste(pil_img,(0,0))
    t     = transforms.ToTensor()(pad).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        st  = torch.ones(1,1).to(DEVICE)
        out = torch.clamp(mod(t, st), 0, 1)
    return out[0].cpu().numpy().transpose(1,2,0)[:ih,:iw]

inp_pil = Image.open(TEST_HAZY).convert("RGB")
inp_np  = _load(TEST_HAZY)
enh_np  = _enh(inp_pil, dh_mod)

has_gt  = TEST_CLEAR is not None and os.path.exists(TEST_CLEAR)
gt_np   = _load(TEST_CLEAR) if has_gt else None
if has_gt and gt_np.shape != inp_np.shape:
    iw2,ih2 = inp_pil.size
    gt_np = np.array(Image.open(TEST_CLEAR).convert("RGB").resize(
                     (iw2,ih2),Image.LANCZOS)).astype(np.float32)/255

# Metrics
if has_gt:
    m_i = {"psnr":psnr_fn(gt_np,inp_np,data_range=1.0),
            "ssim":ssim_fn(gt_np,inp_np,data_range=1.0,channel_axis=2)}
    m_e = {"psnr":psnr_fn(gt_np,enh_np,data_range=1.0),
            "ssim":ssim_fn(gt_np,enh_np,data_range=1.0,channel_axis=2)}
    dp  = m_e["psnr"]-m_i["psnr"]; ds = m_e["ssim"]-m_i["ssim"]

# Strength sweep
sweeps = {}
for sv in [0.25, 0.5, 0.75, 1.0]:
    st = torch.ones(1,1).to(DEVICE)*sv
    iw3,ih3 = inp_pil.size
    pw3,ph3 = ((iw3+3)//4)*4,((ih3+3)//4)*4
    pad3 = Image.new("RGB",(pw3,ph3)); pad3.paste(inp_pil,(0,0))
    t3   = transforms.ToTensor()(pad3).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        o = torch.clamp(dh_mod(t3, torch.ones(1,1).to(DEVICE)*sv),0,1)
    sweeps[sv] = o[0].cpu().numpy().transpose(1,2,0)[:ih3,:iw3]

# Fig 1: comparison
ncols = 3 if has_gt else 2
fig1,ax1 = plt.subplots(1,ncols,figsize=(6*ncols,5))
ax1[0].imshow(np.clip(inp_np,0,1)); ax1[0].axis("off")
t0 = "Input (Hazy)"
if has_gt: t0 += f"\nPSNR={m_i['psnr']:.2f}  SSIM={m_i['ssim']:.4f}"
ax1[0].set_title(t0,fontsize=10)
col = "#1a7a1a" if (has_gt and dp>=0) else ("#cc2222" if has_gt else "black")
ax1[1].imshow(np.clip(enh_np,0,1)); ax1[1].axis("off")
t1 = "Enhanced (Dehazing filter)"
if has_gt: t1 += f"\nPSNR={m_e['psnr']:.2f}  SSIM={m_e['ssim']:.4f}\nΔPSNR={dp:+.2f}  ΔSSIM={ds:+.4f}"
ax1[1].set_title(t1,fontsize=10,color=col)
if has_gt:
    ax1[2].imshow(np.clip(gt_np,0,1)); ax1[2].axis("off")
    ax1[2].set_title("Ground Truth (clear/)",fontsize=10)
fig1.suptitle(f"Dehazing Filter Test — {os.path.basename(TEST_HAZY)}",
              fontsize=12,fontweight="bold")
plt.tight_layout()
p1 = os.path.join(DRIVE_RESULTS,"dehazing_test_comparison.jpg")
fig1.savefig(p1,dpi=130,bbox_inches="tight"); plt.show()
print(f"Saved: {p1}")

# Fig 2: strength sweep
fig2,ax2 = plt.subplots(1,4,figsize=(22,5))
for idx,(sv,out) in enumerate(sorted(sweeps.items())):
    ax2[idx].imshow(np.clip(out,0,1)); ax2[idx].axis("off")
    t = f"Strength = {sv:.2f}"
    if has_gt:
        pp=psnr_fn(gt_np,out,data_range=1.0); ss=ssim_fn(gt_np,out,data_range=1.0,channel_axis=2)
        t+=f"\nPSNR={pp:.2f}  SSIM={ss:.4f}"
    ax2[idx].set_title(t,fontsize=9)
fig2.suptitle("Dehazing Strength Sweep",fontsize=11,fontweight="bold")
plt.tight_layout()
p2=os.path.join(DRIVE_RESULTS,"dehazing_strength_sweep.jpg")
fig2.savefig(p2,dpi=130,bbox_inches="tight"); plt.show()
print(f"Saved: {p2}")

# Fig 3: diff map
diff_amp = np.clip(np.abs(enh_np-inp_np)*5,0,1)
fig3,ax3=plt.subplots(1,3,figsize=(18,5))
ax3[0].imshow(np.clip(inp_np,0,1));  ax3[0].axis("off"); ax3[0].set_title("Input (Hazy)")
ax3[1].imshow(np.clip(enh_np,0,1)); ax3[1].axis("off"); ax3[1].set_title("Enhanced")
im3=ax3[2].imshow(diff_amp,cmap="hot"); ax3[2].axis("off")
ax3[2].set_title("Difference Map (x5)\nBright=haze removed\nDark=unchanged")
plt.colorbar(im3,ax=ax3[2],shrink=0.8)
fig3.suptitle("What the Dehazing filter changed",fontsize=11,fontweight="bold")
plt.tight_layout()
p3=os.path.join(DRIVE_RESULTS,"dehazing_diff_map.jpg")
fig3.savefig(p3,dpi=130,bbox_inches="tight"); plt.show()

print("\n"+"="*50)
print("  Dehazing Filter Test")
print("="*50)
print(f"  Image : {os.path.basename(TEST_HAZY)}")
if has_gt:
    print(f"  PSNR  : {m_i['psnr']:.2f} dB  →  {m_e['psnr']:.2f} dB  ({dp:+.2f})")
    print(f"  SSIM  : {m_i['ssim']:.4f}   →  {m_e['ssim']:.4f}   ({ds:+.4f})")
    print(f"  Status: {'IMPROVED' if dp>0 else 'REGRESSED'}")
print("="*50)
print("If result looks good → proceed to Cell 5 (Rain Removal)")


---
## Cell 5 — Pre-train RainRemoval on Rain100L + DID-MDN + Rain100H

In [ ]:
# ================================================================
# CELL 5 — PRE-TRAIN RainRemoval on Rain100L + DID-MDN + Rain100H
# Shows: live loss/PSNR curves, before/after grid at end
# ================================================================

# ── Load Rain datasets (extract from zip on demand) ──────────
# Rain100L + Rain100H + DID-MDN are extracted together for this cell only.
from dataset_loader_lazy import load_for_filter, cleanup_after_filter, disk_usage
_paths = load_for_filter("rain", WEATHER_DIR)
RAIN100L_ROOT = _paths.get("rain100l")
RAIN100H_ROOT = _paths.get("rain100h")
DIDMDN_ROOT   = _paths.get("didmdn")
disk_usage()

from afb_module import RainRemovalModule
EPOCHS_RN=40; BSZ_RN=8; SZ_RN=256  # increased from 25 — heavy rain needs more epochs

# ── Rain pairing — handles all three naming conventions ─────────
# Rain100L: rain/norain-100x2.png  →  norain/norain-100.png  (strip x2)
# Rain100H: rain/norain-1089.png   →  norain/norain-1089.png (exact match)
# DID-MDN:  train/rain/1.jpg       →  train/clear/1.jpg      (exact match)

class RainPairDataset(Dataset):
    def __init__(self, rain_dir, clean_dir, size=256, augment=False, label="rain"):
        self.pairs=[]; self.size=size; self.aug=augment
        exts=(".jpg",".jpeg",".png")
        # Build clean lookup: stem -> path
        clean_lut={}
        for f in os.listdir(clean_dir):
            if f.lower().endswith(exts):
                clean_lut[os.path.splitext(f)[0]]=os.path.join(clean_dir,f)
        for f in sorted(os.listdir(rain_dir)):
            if not f.lower().endswith(exts): continue
            stem=os.path.splitext(f)[0]; gt=None
            # Try exact match first
            if stem in clean_lut: gt=clean_lut[stem]
            # Strip x2 suffix (Rain100L)
            elif stem.endswith("x2") and stem[:-2] in clean_lut: gt=clean_lut[stem[:-2]]
            elif stem.endswith("_x2") and stem[:-3] in clean_lut: gt=clean_lut[stem[:-3]]
            # Strip any xN suffix
            else:
                import re as _re
                s2=_re.sub(r'x[0-9]+$','',stem)
                if s2 in clean_lut: gt=clean_lut[s2]
            if gt: self.pairs.append((os.path.join(rain_dir,f),gt))
        print(f"  {label}: {len(self.pairs)} pairs  ({len(clean_lut)} clean imgs)")
        if len(self.pairs)==0:
            r_samples=[os.path.splitext(f)[0] for f in sorted(os.listdir(rain_dir))[:4]]
            c_samples=list(clean_lut.keys())[:4]
            print(f"  WARN 0 pairs! rain stems={r_samples}  clean stems={c_samples}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        ip,gp=self.pairs[idx]
        inp=Image.open(ip).convert("RGB"); gt=Image.open(gp).convert("RGB")
        if self.aug:
            iw,ih=inp.size
            if iw>self.size and ih>self.size:
                x=random.randint(0,iw-self.size); y=random.randint(0,ih-self.size)
                inp=inp.crop((x,y,x+self.size,y+self.size))
                gt =gt.crop( (x,y,x+self.size,y+self.size))
            else:
                inp=inp.resize((self.size,self.size),Image.LANCZOS)
                gt =gt.resize( (self.size,self.size),Image.LANCZOS)
            if random.random()>0.5:
                inp=inp.transpose(Image.FLIP_LEFT_RIGHT)
                gt =gt.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            inp=inp.resize((self.size,self.size),Image.LANCZOS)
            gt =gt.resize( (self.size,self.size),Image.LANCZOS)
        return {"input":transforms.ToTensor()(inp),"target":transforms.ToTensor()(gt)}

def _rain_dirs(root):
    """Walk to find rain/ and norain/clean/clear/ sibling dirs."""
    for r,dirs,_ in os.walk(root):
        if r[len(root):].count(os.sep)>3: continue
        dc={d.lower():d for d in dirs}
        if "rain" in dc:
            for cn in ["norain","clean","clear","gt"]:
                if cn in dc:
                    return os.path.join(r,dc["rain"]),os.path.join(r,dc[cn])
    raise FileNotFoundError(f"rain+clean not found under {root}")

rain_ds=[]; rain_wt=[]
for root,label,w in [(RAIN100L_ROOT,"Rain100L",1.0),
                     (RAIN100H_ROOT,"Rain100H",1.5)]:
    if not root or not os.path.isdir(root): continue
    try:
        rd,cd=_rain_dirs(root)
        print(f"  {label}: rain={os.path.basename(rd)}  clean={os.path.basename(cd)}")
        ds=RainPairDataset(rd,cd,SZ_RN,augment=True,label=label)
        rain_ds.append(ds); rain_wt.extend([w]*len(ds))
    except Exception as e: print(f"  {label} skipped: {e}")

# DID-MDN has fixed structure: medium_dataset/train/rain + train/clear
if DIDMDN_ROOT and os.path.isdir(DIDMDN_ROOT):
    try:
        rd=os.path.join(DIDMDN_ROOT,"train","rain")
        cd=os.path.join(DIDMDN_ROOT,"train","clear")
        print(f"  DID-MDN: rain={rd}  clean={cd}")
        ds=RainPairDataset(rd,cd,SZ_RN,augment=True,label="DID-MDN")
        rain_ds.append(ds); rain_wt.extend([1.2]*len(ds))
    except Exception as e: print(f"  DID-MDN skipped: {e}")

assert rain_ds,"No rain datasets found!"
combined=ConcatDataset(rain_ds)
sampler=WeightedRandomSampler(torch.tensor(rain_wt,dtype=torch.float),len(combined),True)
print(f"  Combined: {len(combined)} pairs total")

# Val: DID-MDN test split (has separate test/rain + test/clean)
va_ds=None
if DIDMDN_ROOT:
    try:
        vrd=os.path.join(DIDMDN_ROOT,"test","rain")
        vcd=os.path.join(DIDMDN_ROOT,"test","clean")
        if os.path.isdir(vrd) and os.path.isdir(vcd):
            va_ds=RainPairDataset(vrd,vcd,SZ_RN,augment=False,label="DID-MDN-test")
    except Exception as e: print(f"  DID-MDN val: {e}")
if va_ds is None or len(va_ds)==0:
    # Fallback: use first 200 of training data as val
    va_ds=torch.utils.data.Subset(rain_ds[0],range(min(200,len(rain_ds[0]))))
    print("  Val: using 200-sample subset of Rain100L train")

tr_ld=DataLoader(combined,batch_size=BSZ_RN,sampler=sampler,num_workers=2,pin_memory=True)
va_ld=DataLoader(va_ds,  batch_size=BSZ_RN,shuffle=False,   num_workers=2,pin_memory=True)
print(f"  Train: {len(tr_ld)} batches  Val: {len(va_ld)} batches")
rr_mod=RainRemovalModule()
rr_mod=train_filter_viz(rr_mod,tr_ld,va_ld,"rainremoval",
                        n_epochs=EPOCHS_RN,lr=2e-4,color="#EF9F27")
print("\nBefore/After on validation set:")
before_after_grid(rr_mod,va_ds,n=6,title="Rain Removal: Input | Enhanced | GT")



# Optional: free disk before next cell
# cleanup_after_filter("rain")        # frees Rain100L+H+DID-MDN (~1.5 GB) before SD1


---
## Cell 6 — Pre-train GlareReduction on SD1

In [ ]:
# ================================================================
# CELL 6 — PRE-TRAIN GlareReduction on SD1
# Shows: live loss/PSNR curves, glare map overlay, before/after
# ================================================================

# ── Load SD1 dataset (extract from zip on demand) ─────────────
# SD1 is 10 GB. It extracts to local /content/datasets/ (not Drive).
# After training it will be deleted automatically to free space.
#
# If SD1.zip is on a SECOND Google Drive account:
#   from dataset_loader import mount_second_drive
#   mount_second_drive("/content/drive2")
#   _paths = load_for_filter("glare", WEATHER_DIR,
#                sd1_source="/content/drive2/MyDrive/SD1.zip")
#
from dataset_loader_lazy import load_for_filter, cleanup_after_filter, disk_usage
_paths = load_for_filter("glare", WEATHER_DIR)
SD1_ROOT = _paths.get("sd1")
disk_usage()   # shows how much disk is in use before 10 GB extraction

from afb_module import GlareReductionModule
EPOCHS_GL=30; BSZ_GL=8; SZ_GL=256

def glare_loss_fn(pred,tgt,batch):
    loss=base_loss(pred,tgt)
    if "glare_map" in batch:
        gm=batch["glare_map"].to(DEVICE)
        mask=(gm>0.3).float()
        loss=loss+((pred-tgt).abs()*mask*3.0).mean()
    return loss

def _sd_subdir(names):
    for n in names:
        p=os.path.join(SD1_ROOT,n)
        if os.path.isdir(p): return p
    return SD1_ROOT

if _exists(SD1_ROOT):
    tr_d=_sd_subdir(["train","Train"])
    va_d=_sd_subdir(["val","Val"])
    te_d=_sd_subdir(["test","Test"])

    tr_ds=SD1DS(tr_d,SZ_GL,augment=True)
    if va_d!=SD1_ROOT and va_d!=tr_d:
        va_ds=SD1DS(va_d,SZ_GL,augment=False)
    elif te_d!=SD1_ROOT:
        va_ds=SD1TestDS(te_d,SZ_GL)
    else:
        n_va=max(1,len(tr_ds)//5); n_tr=len(tr_ds)-n_va
        tr_ds,va_ds=torch.utils.data.random_split(tr_ds,[n_tr,n_va],
                    generator=torch.Generator().manual_seed(42))
        print(f"  80/20 split: {n_tr} train / {n_va} val")

    tr_ld=DataLoader(tr_ds,batch_size=BSZ_GL,shuffle=True, num_workers=2,pin_memory=True)
    va_ld=DataLoader(va_ds,batch_size=BSZ_GL,shuffle=False,num_workers=2,pin_memory=True)

    gl_mod=GlareReductionModule()
    gl_mod=train_filter_viz(gl_mod,tr_ld,va_ld,"glare",
                            n_epochs=EPOCHS_GL,lr=1e-4,
                            loss_fn=glare_loss_fn,color="#D85A30")

    # Show glare map overlay visualization
    print("\nGlare map overlay visualization:")
    gl_mod.eval(); sample_batch=next(iter(va_ld))
    inp=sample_batch["input"][:4].to(DEVICE)
    tgt=sample_batch["target"][:4].to(DEVICE)
    with torch.no_grad():
        if "glare_map" in sample_batch:
            gm=sample_batch["glare_map"][:4].to(DEVICE)
            pred=torch.clamp(gl_mod(inp,torch.ones(4,1).to(DEVICE),gm),0,1)
            gm_np=gm[0,0].cpu().numpy()
        else:
            pred=torch.clamp(gl_mod(inp,torch.ones(4,1).to(DEVICE)),0,1)
            gm_np=None

    fig,axes=plt.subplots(3,4,figsize=(14,9)) if gm_np is not None else plt.subplots(2,4,figsize=(14,6))
    for i in range(4):
        in_np =np.clip(inp[i].cpu().numpy().transpose(1,2,0),0,1)
        pr_np =np.clip(pred[i].cpu().numpy().transpose(1,2,0),0,1)
        gt_np =np.clip(tgt[i].cpu().numpy().transpose(1,2,0),0,1)
        psnr_in =psnr_fn(gt_np,in_np, data_range=1.0)
        psnr_out=psnr_fn(gt_np,pr_np,data_range=1.0)
        axes[0,i].imshow(in_np);  axes[0,i].axis("off"); axes[0,i].set_title(f"Input\n{psnr_in:.1f}dB",fontsize=8)
        axes[1,i].imshow(pr_np);  axes[1,i].axis("off"); axes[1,i].set_title(f"Enhanced\n{psnr_out:.1f}dB",fontsize=8,color="#1a7a1a" if psnr_out>psnr_in else "#cc2222")
        if gm_np is not None and axes.shape[0]>2:
            gm_i=sample_batch["glare_map"][i,0].numpy() if "glare_map" in sample_batch else np.zeros((SZ_GL,SZ_GL))
            axes[2,i].imshow(gm_i,cmap="hot",vmin=0,vmax=1); axes[2,i].axis("off"); axes[2,i].set_title("Glare Map",fontsize=8)
    row_labels=["Input","Enhanced","Glare Map"] if gm_np is not None else ["Input","Enhanced"]
    for ri,rl in enumerate(row_labels): axes[ri,0].set_ylabel(rl,fontsize=9,fontweight="bold")
    plt.suptitle("GlareReduction: Input | Enhanced | Glare Map",fontsize=11,fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("SD1 not found. Check SD1_ROOT.")

# Optional: free disk before next cell
# cleanup_after_filter("glare")       # frees SD1 (~10 GB) before WTT loads


---
## Cell 7 — Pre-train IllumNorm on WTT Synthetic

In [ ]:
# ================================================================
# CELL 7 — PRE-TRAIN IllumNorm on WTT Synthetic
# Shows: live loss/PSNR curves, before/after grid at end
# ================================================================

# ── Load WTT Synthetic dataset (extract from zip on demand) ──
from dataset_loader_lazy import load_for_filter, cleanup_after_filter, disk_usage
_paths = load_for_filter("illumnorm", WEATHER_DIR)
DATA_LOCAL = _paths.get("wtt", DATA_LOCAL)

from afb_module import IlluminationNormalizationModule
EPOCHS_IN=20; BSZ_IN=8; SZ_IN=256

if _exists(DATA_LOCAL):
    tr_ds=WTTDS(DATA_LOCAL,"train",SZ_IN)
    va_ds=WTTDS(DATA_LOCAL,"val",  SZ_IN)
    if len(tr_ds)>0 and len(va_ds)>0:
        tr_ld=DataLoader(tr_ds,batch_size=BSZ_IN,shuffle=True, num_workers=2,pin_memory=True)
        va_ld=DataLoader(va_ds,batch_size=BSZ_IN,shuffle=False,num_workers=2,pin_memory=True)
        in_mod=IlluminationNormalizationModule()
        in_mod=train_filter_viz(in_mod,tr_ld,va_ld,"illumnorm",
                               n_epochs=EPOCHS_IN,lr=2e-4,color="#7F77DD")
        print("\nBefore/After on validation set:")
        before_after_grid(in_mod,va_ds,n=6,title="IllumNorm: Input | Enhanced | GT")
    else:
        print("WTT dataset too small. Check DATA_LOCAL.")
else:
    print("DATA_LOCAL not found.")

# Optional: free disk before next cell
# cleanup_after_filter("illumnorm")   # frees WTT before fine-tuning


---
## Cell 8 — Assemble Full Model

In [ ]:
# ================================================================
# CELL 8 — ASSEMBLE: load pre-trained filters into full model
# Shows: which filters loaded, EPE health, sample forward pass
# ================================================================
from complete_model import AdaptiveEnhancementModel
from config import CONFIG, load_config, save_config

save_config(CONFIG, os.path.join(COLAB_ROOT,"config.yaml"))
config=load_config(os.path.join(COLAB_ROOT,"config.yaml"))

MODEL=AdaptiveEnhancementModel(
    backbone=config["model"]["backbone"],
    num_weather_classes=config["model"]["num_weather_classes"],
    num_time_classes=config["model"]["num_time_classes"],
    num_illum_classes=config["model"]["num_illum_classes"],
    feature_dim=config["model"]["feature_dim"],
    num_filters=config["model"]["num_filters"],
    pretrained=True,
).to(DEVICE)

# Load pre-trained filter weights
print("Loading pre-trained filters:")
loaded=[]; filter_files=[(0,"lowlight_best"),(1,"dehazing_best"),(2,"rainremoval_best"),(3,"illumnorm_best"),(4,"glare_best")]
for idx,sname in filter_files:
    p=os.path.join(DRIVE_PRE,f"{sname}.pth")
    if os.path.exists(p):
        MODEL.afb.filters[idx].load_state_dict(torch.load(p,map_location=DEVICE))
        loaded.append(idx); print(f"  [filter {idx}] {FILTER_NAMES[idx]:<22} OK")
    else:
        print(f"  [filter {idx}] {FILTER_NAMES[idx]:<22} NOT FOUND - random init")

# Load EPE from existing checkpoint
ex=find_latest(DRIVE_CKPTS)
if ex:
    ck=torch.load(ex,map_location=DEVICE)
    epe_state={k[4:]:v for k,v in ck["model_state_dict"].items() if k.startswith("epe.")}
    MODEL.epe.load_state_dict(epe_state,strict=False)
    print(f"\nEPE loaded from {os.path.basename(ex)} (epoch {ck['epoch']})")

# Save assembled
assembled=os.path.join(DRIVE_CKPTS,"assembled_pretrained.pth")
torch.save({"epoch":-1,"model_state_dict":MODEL.state_dict(),
            "best_val_enh":float("inf"),"config":config,
            "version":"v5_assembled","loaded_filters":loaded},assembled)
print(f"Assembled model saved: {assembled}")

# Health check: forward pass on random input
MODEL.eval()
with torch.no_grad():
    x=torch.rand(4,3,256,256).to(DEVICE)
    out=MODEL(x)
    fw=out["filter_weights"]
    ent=out["weight_entropy"].mean().item()
    tau=MODEL.ess.tau.item()

# Visualize initial filter weight distribution
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
fw_mean=fw.mean(0).cpu().numpy()
a1.bar(FILTER_NAMES,fw_mean,color=FILTER_COLORS,edgecolor="white",width=0.6)
a1.set_title("Initial Filter Weights (random input)",fontsize=11)
a1.set_ylabel("Mean Weight"); a1.set_ylim(0,1)
a1.axhline(1/5,color="gray",ls="--",lw=1,label="Uniform 0.20")
a1.legend(); plt.setp(a1.xaxis.get_majorticklabels(),rotation=15)

# Filter param counts
param_counts=[sum(p.numel() for p in MODEL.afb.filters[i].parameters()) for i in range(5)]
a2.bar(FILTER_NAMES,param_counts,color=FILTER_COLORS,edgecolor="white",width=0.6)
a2.set_title("Filter Parameter Counts",fontsize=11); a2.set_ylabel("Parameters")
plt.setp(a2.xaxis.get_majorticklabels(),rotation=15)
for i,(bar,n) in enumerate(zip(a2.patches,param_counts)):
    a2.text(bar.get_x()+bar.get_width()/2,bar.get_height()+500,f"{n:,}",ha="center",fontsize=8)

plt.suptitle(f"Assembled Model Health  |  τ={tau:.3f}  H={ent:.3f}  {len(loaded)}/5 filters pre-trained",
             fontsize=11,fontweight="bold")
plt.tight_layout(); plt.show()

total=sum(p.numel() for p in MODEL.parameters())
print(f"\nTotal parameters: {total:,}")
print(f"τ={tau:.4f}  H={ent:.4f}  ({len(loaded)}/5 filters pre-trained)")
print(f"Status: {'OK' if 0.2<tau<1.5 else 'WARNING tau'}")


---
## Cell 9 — Joint Fine-tuning

---
## Cell 8b — One-time: Save fine-tuning dataset to Drive
*Run once. Saves 500 images per dataset permanently to Drive. After this, Cell 9 loads directly — no zip extraction on reconnect.*

In [ ]:
# ================================================================
# CELL 8b — ONE-TIME: Extract and save fine-tuning dataset to Drive
# ================================================================
# Run this cell ONCE. It:
#   1. Extracts all zips
#   2. Picks max 500 images from each dataset
#   3. Copies them to Drive permanently as flat folders
#   4. From next session, Cell 9 loads directly from Drive
#      (no zip extraction needed ever again)
#
# After this cell completes successfully, you NEVER need to
# re-extract zips for fine-tuning. Just run Cell 9 directly.
# ================================================================

import os, shutil, random, glob
from PIL import Image
from tqdm.notebook import tqdm

# ── Where to save on Drive ────────────────────────────────────
FT_DATA_DIR = os.path.join(DRIVE_ROOT, "finetune_data")
os.makedirs(FT_DATA_DIR, exist_ok=True)
print(f"Fine-tuning data will be saved to: {FT_DATA_DIR}")

# ── Config ────────────────────────────────────────────────────
MAX_PER_DATASET = 500    # max images per dataset
IMG_SIZE        = 512    # resize all to 512x512 for consistency

# ── Helper: copy paired images ────────────────────────────────
def save_pairs(pairs, out_dir, label, max_n=MAX_PER_DATASET):
    """
    Save up to max_n paired (input, target) images to out_dir.
    out_dir/
        input/   degraded images
        target/  clean GT images
    """
    inp_dir = os.path.join(out_dir, "input")
    tgt_dir = os.path.join(out_dir, "target")
    os.makedirs(inp_dir, exist_ok=True)
    os.makedirs(tgt_dir, exist_ok=True)

    # Check how many already saved
    existing = len([f for f in os.listdir(inp_dir)
                    if f.lower().endswith((".jpg",".png",".jpeg"))])
    if existing >= min(len(pairs), max_n):
        print(f"  {label}: already saved ({existing} pairs) — skipping")
        return existing

    selected = random.sample(pairs, min(len(pairs), max_n))
    saved = 0
    for idx, (ip, gp) in enumerate(tqdm(selected, desc=f"  {label}", leave=False)):
        try:
            fname = f"{label}_{idx:04d}.jpg"
            # Input
            img = Image.open(ip).convert("RGB").resize((IMG_SIZE,IMG_SIZE), Image.LANCZOS)
            img.save(os.path.join(inp_dir, fname), quality=92)
            # Target
            gt  = Image.open(gp).convert("RGB").resize((IMG_SIZE,IMG_SIZE), Image.LANCZOS)
            gt.save(os.path.join(tgt_dir, fname), quality=92)
            saved += 1
        except Exception as e:
            pass
    print(f"  {label}: saved {saved} pairs → {out_dir}")
    return saved

# ── Extract datasets on-demand ────────────────────────────────
from dataset_loader_lazy import load_for_filter, _resolve_lol, _resolve_reside, _resolve_rain, _resolve_didmdn
from dataset_real_pairs import _find_rain_dirs
import re as _re

print("\n" + "="*60)
print("  Extracting datasets (one time only)...")
print("="*60)

all_pairs = {}  # label -> list of (inp_path, tgt_path)

# ── LOL ────────────────────────────────────────────────────────
try:
    ft = load_for_filter("finetune", WEATHER_DIR)
    lol_r = ft.get("lol")
    if lol_r and os.path.isdir(lol_r):
        # Train split
        for root, dirs, _ in os.walk(lol_r):
            dc = {d.lower():d for d in dirs}
            if "low" in dc and "high" in dc:
                low_d = os.path.join(root, dc["low"])
                high_d= os.path.join(root, dc["high"])
                exts  = (".jpg",".png",".jpeg")
                pairs = []
                for f in os.listdir(low_d):
                    if not f.lower().endswith(exts): continue
                    stem = os.path.splitext(f)[0]
                    for ext in exts:
                        gp = os.path.join(high_d, stem+ext)
                        if os.path.exists(gp):
                            pairs.append((os.path.join(low_d,f), gp)); break
                if pairs:
                    all_pairs["LOL"] = pairs
                    print(f"  LOL: {len(pairs)} pairs found")
                break
except Exception as e:
    print(f"  LOL: {e}")

# ── WTT ────────────────────────────────────────────────────────
try:
    wtt_r = ft.get("wtt")
    if wtt_r and os.path.isdir(wtt_r):
        for split in ["train"]:
            id_ = os.path.join(wtt_r, split, "images")
            cd_ = os.path.join(wtt_r, split, "clean_images")
            if os.path.isdir(id_) and os.path.isdir(cd_):
                pairs = []
                for f in os.listdir(id_):
                    if not f.lower().endswith((".jpg",".png",".jpeg")): continue
                    gp = os.path.join(cd_, f)
                    if os.path.exists(gp):
                        pairs.append((os.path.join(id_,f), gp))
                if pairs:
                    all_pairs["WTT"] = pairs
                    print(f"  WTT: {len(pairs)} pairs found")
except Exception as e:
    print(f"  WTT: {e}")

# ── Rain100L ───────────────────────────────────────────────────
try:
    r100l = ft.get("rain100l")
    if r100l and os.path.isdir(r100l):
        rd, cd = _find_rain_dirs(r100l)
        if rd and cd:
            pairs = []
            clean_lut = {os.path.splitext(f)[0]: os.path.join(cd,f)
                         for f in os.listdir(cd)
                         if f.lower().endswith((".jpg",".png",".jpeg"))}
            for f in os.listdir(rd):
                if not f.lower().endswith((".jpg",".png",".jpeg")): continue
                stem = os.path.splitext(f)[0]
                gt = clean_lut.get(stem) or clean_lut.get(
                     stem[:-2] if stem.endswith("x2") else None) or                      clean_lut.get(_re.sub(r"x[0-9]+$","",stem))
                if gt: pairs.append((os.path.join(rd,f), gt))
            if pairs:
                all_pairs["Rain100L"] = pairs
                print(f"  Rain100L: {len(pairs)} pairs found")
except Exception as e:
    print(f"  Rain100L: {e}")

# ── RESIDE-ITS ─────────────────────────────────────────────────
try:
    dh = load_for_filter("dehazing", WEATHER_DIR)
    reside_r = dh.get("reside")
    if reside_r and os.path.isdir(reside_r):
        hazy_d = clear_d = None
        for r, dirs, _ in os.walk(reside_r):
            dc = {d.lower():d for d in dirs}
            if "hazy" in dc and "clear" in dc:
                hazy_d = os.path.join(r, dc["hazy"])
                clear_d= os.path.join(r, dc["clear"]); break
        if hazy_d and clear_d:
            clear_lut = {os.path.splitext(f)[0]: os.path.join(clear_d,f)
                         for f in os.listdir(clear_d)
                         if f.lower().endswith((".jpg",".png",".jpeg"))}
            pairs = []
            for f in os.listdir(hazy_d):
                if not f.lower().endswith((".jpg",".png",".jpeg")): continue
                gt_stem = os.path.splitext(f)[0].split("_")[0]
                if gt_stem in clear_lut:
                    pairs.append((os.path.join(hazy_d,f), clear_lut[gt_stem]))
            if pairs:
                all_pairs["RESIDE"] = pairs
                print(f"  RESIDE: {len(pairs)} pairs found")
except Exception as e:
    print(f"  RESIDE: {e}")

# ── Rain100H + DID-MDN ─────────────────────────────────────────
try:
    rn = load_for_filter("rain", WEATHER_DIR)
    for key, label in [("rain100h","Rain100H"),("didmdn","DIDMDN")]:
        root = rn.get(key)
        if not root or not os.path.isdir(root): continue
        if label == "Rain100H":
            rd, cd = _find_rain_dirs(root)
        else:
            rd = os.path.join(root, "train", "rain")
            cd = os.path.join(root, "train", "clear")
        if rd and cd and os.path.isdir(rd) and os.path.isdir(cd):
            clean_lut = {os.path.splitext(f)[0]: os.path.join(cd,f)
                         for f in os.listdir(cd)
                         if f.lower().endswith((".jpg",".png",".jpeg"))}
            pairs = [(os.path.join(rd,f), clean_lut[os.path.splitext(f)[0]])
                     for f in os.listdir(rd)
                     if f.lower().endswith((".jpg",".png",".jpeg"))
                     and os.path.splitext(f)[0] in clean_lut]
            if pairs:
                all_pairs[label] = pairs
                print(f"  {label}: {len(pairs)} pairs found")
except Exception as e:
    print(f"  Rain/DID-MDN: {e}")

# ── SD1 ────────────────────────────────────────────────────────
try:
    gl = load_for_filter("glare", WEATHER_DIR)
    sd1_r = gl.get("sd1")
    if sd1_r and os.path.isdir(sd1_r):
        # Use 3-panel strips
        pairs = []
        for root, _, files in os.walk(sd1_r):
            for f in files:
                if f.lower().endswith((".jpg",".png",".bmp",".jpeg")):
                    pairs.append(("strip:" + os.path.join(root,f), None))
            if pairs: break
        if pairs:
            all_pairs["SD1"] = pairs
            print(f"  SD1: {len(pairs)} strips found")
except Exception as e:
    print(f"  SD1: {e}")

# ── Save all pairs to Drive ───────────────────────────────────
print("\n" + "="*60)
print("  Saving to Drive (this is the one-time step)...")
print("="*60)

DATASET_WEIGHTS = {
    "LOL":      3.0,
    "WTT":      1.0,
    "Rain100L": 1.0,
    "Rain100H": 1.5,
    "DIDMDN":   1.2,
    "RESIDE":   1.5,
    "SD1":      2.0,
}

import json as _json
manifest = {}   # label -> {n_pairs, weight, inp_dir, tgt_dir}

for label, pairs in all_pairs.items():
    out_dir = os.path.join(FT_DATA_DIR, label)

    if label == "SD1":
        # Handle strips separately
        inp_dir = os.path.join(out_dir, "input")
        tgt_dir = os.path.join(out_dir, "target")
        os.makedirs(inp_dir, exist_ok=True)
        os.makedirs(tgt_dir, exist_ok=True)
        existing = len([f for f in os.listdir(inp_dir)
                        if f.lower().endswith((".jpg",".png"))])
        if existing >= MAX_PER_DATASET:
            print(f"  SD1: already saved ({existing}) — skipping")
            n = existing
        else:
            selected = random.sample(pairs, min(len(pairs), MAX_PER_DATASET))
            n = 0
            for idx, (sp, _) in enumerate(tqdm(selected, desc="  SD1", leave=False)):
                try:
                    img = Image.open(sp[6:]).convert("RGB")
                    W, H = img.size; pw = W//3
                    fname = f"SD1_{idx:04d}.jpg"
                    gt_pil  = img.crop((0,0,pw,H)).resize((IMG_SIZE,IMG_SIZE),Image.LANCZOS)
                    inp_pil = img.crop((pw,0,pw*2,H)).resize((IMG_SIZE,IMG_SIZE),Image.LANCZOS)
                    inp_pil.save(os.path.join(inp_dir, fname), quality=92)
                    gt_pil.save( os.path.join(tgt_dir, fname), quality=92)
                    n += 1
                except: pass
            print(f"  SD1: saved {n} pairs")
    else:
        n = save_pairs(pairs, out_dir, label)

    manifest[label] = {
        "n_pairs": n,
        "weight":  DATASET_WEIGHTS.get(label, 1.0),
        "inp_dir": os.path.join(out_dir, "input"),
        "tgt_dir": os.path.join(out_dir, "target"),
    }

# Save manifest
manifest_path = os.path.join(FT_DATA_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    _json.dump(manifest, f, indent=2)

print("\n" + "="*60)
print("  DONE — Fine-tuning data saved to Drive")
print("="*60)
total = sum(v["n_pairs"] for v in manifest.values())
for label, info in manifest.items():
    print(f"  {label:<12} {info['n_pairs']:>4} pairs   w={info['weight']}  → {info['inp_dir']}")
print(f"  {'TOTAL':<12} {total:>4} pairs")
print("="*60)
print("\nNext time: Cell 9 loads directly from Drive.")
print("No zip extraction needed ever again.")
print(f"\nManifest saved: {manifest_path}")


In [ ]:
# ================================================================
# CELL 9 — JOINT FINE-TUNING
# Live dashboard: train loss, val loss, H entropy, tau, W-accuracy
# Phase A (0-10): ESS only   Phase B (10-40): all modules
# Kill and re-run to resume from latest checkpoint
# ================================================================
from complete_model import AdaptiveEnhancementModel
from losses import EPELoss, FullPipelineLoss
from dataset_real_pairs import create_combined_dataloaders
from config import load_config, save_config, CONFIG
from torch.utils.tensorboard import SummaryWriter

FINETUNE_EPOCHS = 60   # extended from 40 — compound conditions need more fine-tuning
PHASE_B_START   = 10   # absolute epoch — Phase B from epoch 10 onwards
BATCH_SIZE      = 4
KEEP_N          = 5

config=load_config(os.path.join(COLAB_ROOT,"config.yaml"))

# ── Load fine-tuning data from Drive (no zip extraction) ──────────
# Run Cell 8b ONCE to create this folder.
# After that, every session loads instantly from Drive.
import os, json as _json

FT_DATA_DIR   = os.path.join(DRIVE_ROOT, "finetune_data")
MANIFEST_PATH = os.path.join(FT_DATA_DIR, "manifest.json")

if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH) as _f:
        FT_MANIFEST = _json.load(_f)
    print("Fine-tuning data loaded from Drive:")
    _total = 0
    for _label, _info in FT_MANIFEST.items():
        _ok = os.path.isdir(_info["inp_dir"])
        print(f"  {'OK' if _ok else 'MISS':<6} {_label:<12} {_info['n_pairs']:>4} pairs  w={_info['weight']}")
        if _ok: _total += _info["n_pairs"]
    print(f"  Total: {_total} pairs")
else:
    print("ERROR: Fine-tuning data not found.")
    print(f"  Expected: {MANIFEST_PATH}")
    print("  --> Run Cell 8b first (one-time, ~20 min)")
    raise FileNotFoundError("Run Cell 8b first.")


# ── Fix LOL split: resolve train and eval dirs separately ─────────
# LOL resolves to .../LOL/train which is correct for training.
# For val/test we need .../LOL/eval (15 pairs from eval15).
import os as _os
LOL_TRAIN_ROOT = LOL_ROOT   # e.g. /content/datasets/LOL/LOL/train
LOL_EVAL_ROOT  = None
if LOL_ROOT:
    # Walk up to find eval/ or eval15/ sibling
    parent = _os.path.dirname(LOL_ROOT)   # .../LOL/LOL
    for sub in ["eval","eval15","test"]:
        ep = _os.path.join(parent, sub)
        if _os.path.isdir(ep):
            # Check it has low/ and high/
            ec = {d.lower() for d in _os.listdir(ep)}
            if "low" in ec and "high" in ec:
                LOL_EVAL_ROOT = ep
                print(f"  LOL eval dir: {LOL_EVAL_ROOT}")
                break
    if LOL_EVAL_ROOT is None:
        print("  LOL eval dir not found — using train for val (not ideal)")
        LOL_EVAL_ROOT = LOL_TRAIN_ROOT

# ── Limit WTT to keep epochs fast ─────────────────────────────────
# Full WTT (8800 train) = 28 min/epoch = 18+ hours for 40 epochs.
# Use 20% subset = ~1760 images = ~6 min/epoch = 4 hours total.
# Set WTT_SUBSET_FRAC = 1.0 to use full dataset.
# ── Dataset subset fractions (adjust to control epoch time) ────────
# With all datasets included, full epoch would take ~2 hours.
# These fractions keep it manageable (~15-20 min/epoch).
WTT_SUBSET_FRAC    = 0.20   # WTT:    8800 x 0.20 = 1760 images
RESIDE_SUBSET_FRAC = 0.50   # RESIDE: 13990 x 0.50 = 6995 images
SD1_SUBSET_FRAC    = 0.20   # SD1:    ~12000 x 0.20 = 2400 images
# Approximate total training images per epoch:
# 1760 (WTT) + 485 (LOL) + 200 (R100L) + 1255 (R100H) + 4000 (DID-MDN) + 6995 (RESIDE) + 2400 (SD1) ≈ 17,095

# ── Also extract Rain100H for fine-tuning (heavy rain coverage) ───
_rain100h_path = None
if RAIN100H_ROOT and _os.path.isdir(RAIN100H_ROOT):
    _rain100h_path = RAIN100H_ROOT
else:
    # Try to extract from zip
    try:
        _rh = load_for_filter.__module__
        from dataset_loader_lazy import load_for_filter as _lff
        _rh_paths = _lff("rain", WEATHER_DIR)
        _rain100h_path = _rh_paths.get("rain100h")
        RAIN100H_ROOT  = _rain100h_path
        print(f"  Rain100H for fine-tuning: {_rain100h_path}")
    except Exception as _e:
        print(f"  Rain100H not available: {_e}")

# ── Build dataloaders from saved Drive dataset ────────────────────
print("\nBuilding dataloaders...")
from dataset_real_pairs import ManifestDataLoader

dataloaders = ManifestDataLoader(
    manifest      = FT_MANIFEST,
    batch_size    = BATCH_SIZE,
    num_workers   = 2,
    image_size    = config["data"]["image_size"],
    debug         = True,
).get_loaders()

assert "train" in dataloaders, (
    "No training data. Check Cell 8b ran successfully and "
    f"{FT_DATA_DIR} exists on Drive."
)
print(f"  Train batches : {len(dataloaders['train'])}")
print(f"  Val batches   : {len(dataloaders.get('val', []))}")


MODEL=AdaptiveEnhancementModel(
    backbone=config["model"]["backbone"],
    num_weather_classes=config["model"]["num_weather_classes"],
    num_time_classes=config["model"]["num_time_classes"],
    num_illum_classes=config["model"]["num_illum_classes"],
    feature_dim=config["model"]["feature_dim"],
    num_filters=config["model"]["num_filters"],
    pretrained=False,
).to(DEVICE)

# ── Verify architecture matches checkpoint before loading ────────────
# The checkpoint may have been saved with an older/different AFB architecture.
# Load with strict=False to skip mismatched keys, then report what was loaded.
def _safe_load(model, state_dict):
    result = model.load_state_dict(state_dict, strict=False)
    missing   = [k for k in result.missing_keys   if not any(x in k for x in ["num_batches","running"])]
    unexpected= [k for k in result.unexpected_keys if not any(x in k for x in ["num_batches","running"])]
    if missing:
        print(f"  Keys in model but NOT in checkpoint ({len(missing)}):")
        for k in missing[:6]: print(f"    {k}")
        if len(missing)>6: print(f"    ... and {len(missing)-6} more")
        print("  These will use random init — OK if they are new layers.")
    if unexpected:
        print(f"  Keys in checkpoint but NOT in model ({len(unexpected)}):")
        for k in unexpected[:6]: print(f"    {k}")
        if len(unexpected)>6: print(f"    ... and {len(unexpected)-6} more")
        print("  These are old layers — will be ignored.")
    n_match = len(state_dict) - len(unexpected)
    print(f"  Matched {n_match}/{len(state_dict)} checkpoint keys.")
    return result

# ── Smart resume: always prefer latest epoch checkpoint ──────────────
# Priority: checkpoint_epoch_XXXX.pth > best_model.pth > assembled_pretrained.pth
# This means: after any disconnect, re-running this cell continues
# from exactly where it left off — never restarts from epoch 0.
start_epoch=0; best_val_enh=float("inf"); _loaded_from=None

import glob as _gl
def _epoch_num(p):
    try: return int(os.path.basename(p).split("_")[-1].split(".")[0])
    except: return -1

# Find latest valid epoch checkpoint
_epoch_ckpts = sorted(
    [p for p in _gl.glob(os.path.join(DRIVE_CKPTS,"checkpoint_epoch_*.pth"))
     if _valid(p)],
    key=_epoch_num, reverse=True
)
_resume_ckpt = _epoch_ckpts[0] if _epoch_ckpts else None

# Fall back to best_model.pth
if _resume_ckpt is None:
    _b = os.path.join(DRIVE_CKPTS,"best_model.pth")
    if _valid(_b): _resume_ckpt = _b

def _load_with_filter_fix(model, state_dict, pretrain_dir):
    """
    Load checkpoint, then reload correct pre-trained filter weights on top.
    This handles the case where the checkpoint was saved with an older/shallower
    AFB architecture — EPE and ESS weights are still reusable.
    """
    # Step 1: load whatever matches (strict=False)
    result = model.load_state_dict(state_dict, strict=False)
    n_loaded = len(state_dict) - len(result.unexpected_keys)
    print(f"  Loaded {n_loaded}/{len(state_dict)} keys from checkpoint")

    # Step 2: if there are size mismatches, reload correct filter weights
    has_mismatch = any("afb.filters" in k for k in result.missing_keys)
    if has_mismatch:
        print("  AFB architecture mismatch detected — reloading pre-trained filters...")
        filter_map = [
            (0, "lowlight_best.pth"),
            (1, "dehazing_best.pth"),
            (2, "rainremoval_best.pth"),
            (3, "illumnorm_best.pth"),
            (4, "glare_best.pth"),
        ]
        reloaded = []
        for idx, fname in filter_map:
            p = os.path.join(pretrain_dir, fname)
            if os.path.exists(p):
                fstate = torch.load(p, map_location=DEVICE)
                try:
                    model.afb.filters[idx].load_state_dict(fstate)
                    reloaded.append(idx)
                except Exception as fe:
                    print(f"    filter[{idx}] {fname}: {fe}")
        print(f"  Reloaded pre-trained filters: {reloaded}")
        print(f"  EPE + ESS weights: loaded from checkpoint")
    return result

if _resume_ckpt is not None:
    ck = torch.load(_resume_ckpt, map_location=DEVICE)
    print(f"Resuming from: {os.path.basename(_resume_ckpt)}")
    _load_with_filter_fix(MODEL, ck["model_state_dict"], DRIVE_PRE)
    best_val_enh = ck.get("best_val_enh", ck.get("best_val_loss", float("inf")))
    start_epoch  = ck["epoch"] + 1
    _loaded_from = "checkpoint"
    print(f"  Last completed epoch : {ck['epoch']}")
    print(f"  Next epoch will be   : {start_epoch}")
    print(f"  Best val_enh so far  : {best_val_enh:.6f}")
    if start_epoch >= FINETUNE_EPOCHS:
        print(f"\nAll {FINETUNE_EPOCHS} epochs already done.")
        print(f"Increase FINETUNE_EPOCHS above {start_epoch} to keep training.")
else:
    assembled = os.path.join(DRIVE_CKPTS,"assembled_pretrained.pth")
    if os.path.exists(assembled):
        ck = torch.load(assembled, map_location=DEVICE)
        print("First run: loading assembled model...")
        _load_with_filter_fix(MODEL, ck["model_state_dict"], DRIVE_PRE)
        _loaded_from = "assembled"
        print(f"  Starting from epoch 0 of {FINETUNE_EPOCHS}")
    else:
        print("WARNING: No checkpoint found. Loading pre-trained filters only.")
        for idx, fname in [(0,"lowlight_best"),(1,"dehazing_best"),(2,"rainremoval_best"),
                           (3,"illumnorm_best"),(4,"glare_best")]:
            p = os.path.join(DRIVE_PRE, f"{fname}.pth")
            if os.path.exists(p):
                MODEL.afb.filters[idx].load_state_dict(torch.load(p,map_location=DEVICE))
                print(f"  filter[{idx}] loaded from {fname}.pth")

criterion=FullPipelineLoss(
    lambda_weather=config["training"].get("lambda_weather",0.5),
    lambda_time   =config["training"].get("lambda_time",   0.3),
    lambda_illum  =config["training"].get("lambda_illum",  0.3),
    lam_text=2.0,lam_ssim=0.3,
)

current_phase = "B" if start_epoch >= PHASE_B_START else "A"
def make_opt(phase):
    global current_phase; current_phase=phase
    if phase=="A":
        for p in MODEL.afb.parameters(): p.requires_grad=False
        for p in MODEL.epe.parameters(): p.requires_grad=False
        for p in MODEL.ess.parameters(): p.requires_grad=True
        for p in MODEL.combiner.parameters(): p.requires_grad=True
        return optim.Adam([p for p in MODEL.parameters() if p.requires_grad],lr=5e-5,weight_decay=1e-5)
    else:
        # Phase B: keep EPE frozen — new fixed labels would destroy 97% accuracy
        for p in MODEL.epe.parameters(): p.requires_grad=False
        for p in MODEL.afb.parameters(): p.requires_grad=True
        for p in MODEL.ess.parameters(): p.requires_grad=True
        for p in MODEL.combiner.parameters(): p.requires_grad=True
        print("  Phase B: AFB+ESS training. EPE frozen.")
        return optim.Adam([p for p in MODEL.parameters() if p.requires_grad],lr=5e-5,weight_decay=1e-5)

optimizer=make_opt("B" if start_epoch >= PHASE_B_START else "A")
scheduler=optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=1,eta_min=1e-6)
writer=SummaryWriter(DRIVE_LOGS)
lf=open(LOG_FILE,"a",buffering=1)
def lg(m): print(m); lf.write(m+"\n"); lf.flush()

lg(f"\n{'='*65}")
lg(f"Joint Fine-tune  {time.strftime('%Y-%m-%d %H:%M')}")
lg(f"Phase A (ESS only) -> Phase B (all) at epoch {PHASE_B_START}")
lg(f"{'='*65}")

# Live dashboard widgets
dash_out=WG.Output()
curve_out=WG.Output()
_D(WG.VBox([
    WG.HTML("<h3 style='color:#1B3A6B'>Joint Fine-tuning Dashboard</h3>"),
    dash_out, curve_out
]))

ep_hist=[]; enh_tr=[]; enh_va=[]; ent_hist=[]; tau_hist=[]; wacc_hist=[]
end_ep=FINETUNE_EPOCHS   # absolute target epoch (not relative)

for epoch in range(start_epoch,end_ep):
    current_phase = "B" if epoch >= PHASE_B_START else "A"
    if epoch==PHASE_B_START and current_phase=="A":
        lg(f"  Switching to Phase B (unfreezing all filters)")
        optimizer=make_opt("B")
        scheduler=optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=1,eta_min=1e-6)

    MODEL.train(); t_epe=t_enh=t_ent=t_n=0
    for batch in tqdm(dataloaders["train"],desc=f"FT Ep{epoch:03d}",leave=True):
        out=MODEL(batch["image"].to(DEVICE))
        loss,bd=criterion(out,batch,DEVICE)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in MODEL.parameters() if p.requires_grad],1.0)
        optimizer.step()
        t_epe+=bd["epe"]; t_enh+=bd["enh"]
        t_ent+=out["weight_entropy"].mean().item(); t_n+=1
    scheduler.step(epoch+1)

    MODEL.eval(); v_enh=v_ent=v_wa=v_n=0
    with torch.no_grad():
        for batch in dataloaders["val"]:
            w_lbl=batch["weather_label"].to(DEVICE)
            out=MODEL(batch["image"].to(DEVICE))
            _,bd=criterion(out,batch,DEVICE)
            v_enh+=bd["enh"]; v_ent+=out["weight_entropy"].mean().item()
            v_wa+=(out["weather_logits"].argmax(1)==w_lbl).float().mean().item()
            v_n+=1

    val_enh=v_enh/v_n; tau=MODEL.ess.tau.item()
    ep_hist.append(epoch); enh_tr.append(t_enh/t_n); enh_va.append(val_enh)
    ent_hist.append(v_ent/v_n); tau_hist.append(tau); wacc_hist.append(v_wa/v_n)

    writer.add_scalars("Loss",{"train":t_enh/t_n,"val":val_enh},epoch)
    writer.add_scalar("Entropy",v_ent/v_n,epoch)
    writer.add_scalar("Tau",tau,epoch)
    writer.add_scalar("W_Acc",v_wa/v_n,epoch)

    is_best=val_enh<best_val_enh
    if is_best: best_val_enh=val_enh

    ckd={"epoch":epoch,"model_state_dict":MODEL.state_dict(),
         "optimizer_state_dict":optimizer.state_dict(),
         "scheduler_state_dict":scheduler.state_dict(),
         "best_val_enh":best_val_enh,"config":config,"version":"v5_finetune"}
    ep_p=os.path.join(DRIVE_CKPTS,f"checkpoint_epoch_{epoch:04d}.pth")
    torch.save(ckd,ep_p)
    if is_best: torch.save(ckd,os.path.join(DRIVE_CKPTS,"best_model.pth"))
    cleanup_old(DRIVE_CKPTS,KEEP_N)

    line=(f"  {epoch:>4}  {current_phase:>2}  epe={t_epe/t_n:.4f}  "
          f"enh={t_enh/t_n:.4f}  H={v_ent/v_n:.3f}  val={val_enh:.4f}  "
          f"W={v_wa/v_n:.3f}  t={tau:.3f}"
          +("  BEST" if is_best else ""))
    lg(line)

    # Update dashboard every epoch
    with dash_out:
        clear_output(wait=True)
        print(f"  Epoch {epoch+1}/{FINETUNE_EPOCHS}  Phase={current_phase}  (resumed from ep {start_epoch})")
        print(f"  Train enh loss  : {t_enh/t_n:.4f}")
        print(f"  Val enh loss    : {val_enh:.4f}  {'BEST' if is_best else ''}")
        print(f"  Weight entropy H: {v_ent/v_n:.4f}  {'OK (blending)' if v_ent/v_n>0.2 else 'WARNING low'}")
        print(f"  Temperature tau : {tau:.4f}  {'OK' if 0.2<tau<1.5 else 'WARNING'}")
        print(f"  Weather accuracy: {v_wa/v_n*100:.1f}%")
        print(f"  Best val_enh    : {best_val_enh:.6f}")

    if len(ep_hist)>=2 and (epoch%2==0 or epoch==end_ep-1):
        with curve_out:
            clear_output(wait=True)
            fig,axes=plt.subplots(2,3,figsize=(15,7))
            axes[0,0].plot(ep_hist,enh_tr,color="#2278CF",lw=2); axes[0,0].plot(ep_hist,enh_va,color="#E8541A",lw=2)
            axes[0,0].legend(["Train","Val"]); axes[0,0].set_title("Enhancement Loss"); axes[0,0].grid(True,alpha=0.3)
            axes[0,1].plot(ep_hist,ent_hist,color="#1D9E75",lw=2)
            axes[0,1].axhline(0.2,color="gray",ls="--",lw=1,label="Min good (0.20)")
            axes[0,1].set_title("Weight Entropy H (higher=better blending)"); axes[0,1].legend(); axes[0,1].grid(True,alpha=0.3)
            axes[0,2].plot(ep_hist,tau_hist,color="#7F77DD",lw=2)
            axes[0,2].axhline(0.2,color="red",ls="--",lw=1,label="Collapse threshold")
            axes[0,2].set_title("Temperature τ"); axes[0,2].legend(); axes[0,2].grid(True,alpha=0.3)
            axes[1,0].plot(ep_hist,[w*100 for w in wacc_hist],color="#EF9F27",lw=2)
            axes[1,0].set_title("Weather Accuracy (%)"); axes[1,0].set_ylim(0,100); axes[1,0].grid(True,alpha=0.3)
            axes[1,1].plot(ep_hist,enh_va,color="#E8541A",lw=2,marker="o",ms=4)
            if best_val_enh<float("inf"): axes[1,1].axhline(best_val_enh,color="green",ls="--",lw=1,label=f"Best {best_val_enh:.4f}")
            axes[1,1].set_title("Val Enhancement Loss"); axes[1,1].legend(); axes[1,1].grid(True,alpha=0.3)
            # Filter weight pie (last batch)
            MODEL.eval()
            with torch.no_grad():
                fw_sample=MODEL(torch.rand(4,3,256,256).to(DEVICE))["filter_weights"].mean(0).cpu().numpy()
            axes[1,2].bar(FILTER_NAMES,fw_sample,color=FILTER_COLORS,edgecolor="white")
            axes[1,2].set_title("Mean Filter Weights (random input)"); axes[1,2].set_ylim(0,1)
            plt.setp(axes[1,2].xaxis.get_majorticklabels(),rotation=15)
            fig.suptitle(f"Fine-tuning Progress  Epoch {epoch}  Phase {current_phase}",fontsize=12,fontweight="bold")
            plt.tight_layout(); plt.show()

# ── Training complete ─────────────────────────────────────────
# Safely read last logged values
_last_H   = ep_list[-1] if ep_list else 0
_last_W   = float(line.split("W=")[1].split()[0]) if "W=" in line else 0
_last_tau = float(line.split("t=")[1].split()[0]) if "t=" in line else 0
_last_H_v = float(line.split("H=")[1].split()[0]) if "H=" in line else 0

_summary = f"""
{'='*65}
  FINE-TUNING COMPLETE
{'='*65}
  Best val_enh      : {best_val_enh:.6f}
  Best checkpoint   : {os.path.join(DRIVE_CKPTS, 'best_model.pth')}
  Total epochs done : {end_ep}
  Last H (entropy)  : {_last_H_v:.4f}  (target > 0.20)
  Last W accuracy   : {_last_W*100:.1f}%
  Last tau          : {_last_tau:.4f}
{'='*65}
  Next steps:
    1. Run Cell 10 (single image test) to verify enhancement
    2. Run Cell 11 (full evaluation) to get PSNR/SSIM/LPIPS
    3. Fill Table 5 in the Scopus paper with results
{'='*65}
"""
lg(_summary)
print(_summary)
lf.close(); writer.close()


---
## Cell 10 — Visual Test: enhance any image
*Random pick from datasets or paste your own path. Re-run for a new random image.*

In [ ]:
# ================================================================
# CELL 10 — VISUAL TEST: Full pipeline on any image
# ================================================================
# MODE 1 — Random from dataset (leave TEST_IMAGE = None):
#   Picks a random degraded image from any available dataset.
#   Ground truth shown automatically if available.
#
# MODE 2 — Your own image (set TEST_IMAGE to a path):
#   TEST_IMAGE = "/content/drive/MyDrive/my_photo.jpg"
#   Works with any JPG/PNG from Drive or local disk.
#
# Run this cell as many times as you like — each run picks a
# different random image. Set DEGRADATION to force a category.
# ================================================================

TEST_IMAGE   = None  # ← paste your image path here, or leave None
DEGRADATION  = None  # ← force: "lowlight" "haze" "rain" "glare" None=auto

import os, random, glob
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as GS
from PIL import Image
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
from complete_model import AdaptiveEnhancementModel
from ess_module import FILTER_NAMES
from config import CONFIG

# ── Load model ────────────────────────────────────────────────
best_ckpt = os.path.join(DRIVE_CKPTS, "best_model.pth")
assert os.path.exists(best_ckpt), f"No checkpoint at {best_ckpt} — run Cell 9 first"

model = AdaptiveEnhancementModel(
    backbone            = CONFIG["model"]["backbone"],
    num_weather_classes = CONFIG["model"]["num_weather_classes"],
    num_time_classes    = CONFIG["model"]["num_time_classes"],
    num_illum_classes   = CONFIG["model"]["num_illum_classes"],
    feature_dim         = CONFIG["model"]["feature_dim"],
    num_filters         = CONFIG["model"]["num_filters"],
    pretrained          = False,
).to(DEVICE)

ck = torch.load(best_ckpt, map_location=DEVICE)
try:    model.load_state_dict(ck["model_state_dict"])
except: model.load_state_dict(ck["model_state_dict"], strict=False)
model.eval()
print(f"Model loaded from {os.path.basename(best_ckpt)}")

# ── Dataset pool for random picks ─────────────────────────────
DATASET_POOL = []   # list of (inp_path, gt_path_or_None, label)

def _add_pairs(inp_dir, gt_dir, label, strip_x2=False, gt_at_first_underscore=False):
    """Scan inp_dir and match GT files."""
    if not inp_dir or not os.path.isdir(inp_dir): return
    exts = (".jpg",".jpeg",".png")
    gt_lut = {}
    if gt_dir and os.path.isdir(gt_dir):
        for f in os.listdir(gt_dir):
            if f.lower().endswith(exts):
                gt_lut[os.path.splitext(f)[0]] = os.path.join(gt_dir, f)
    for f in os.listdir(inp_dir):
        if not f.lower().endswith(exts): continue
        stem = os.path.splitext(f)[0]
        gt = None
        # Try direct match
        if stem in gt_lut: gt = gt_lut[stem]
        # Strip x2 (Rain100L)
        elif strip_x2 and stem.endswith("x2") and stem[:-2] in gt_lut:
            gt = gt_lut[stem[:-2]]
        # Strip suffix at first underscore (RESIDE-ITS)
        elif gt_at_first_underscore:
            s2 = stem.split("_")[0]
            if s2 in gt_lut: gt = gt_lut[s2]
        DATASET_POOL.append((os.path.join(inp_dir, f), gt, label))

# LOL
if LOL_ROOT and os.path.isdir(LOL_ROOT):
    _add_pairs(os.path.join(LOL_ROOT,"low"), os.path.join(LOL_ROOT,"high"), "LOL (low-light)")

# RESIDE
if RESIDE_ROOT and os.path.isdir(RESIDE_ROOT):
    # Walk to find hazy/ and clear/
    for r,dirs,_ in os.walk(RESIDE_ROOT):
        dc = {d.lower():d for d in dirs}
        if "hazy" in dc and "clear" in dc:
            _add_pairs(os.path.join(r,dc["hazy"]), os.path.join(r,dc["clear"]),
                       "RESIDE-ITS (haze)", gt_at_first_underscore=True)
            break

# Rain100L
if RAIN100L_ROOT and os.path.isdir(RAIN100L_ROOT):
    for r,dirs,_ in os.walk(RAIN100L_ROOT):
        dc = {d.lower():d for d in dirs}
        if "rain" in dc and "norain" in dc:
            _add_pairs(os.path.join(r,dc["rain"]), os.path.join(r,dc["norain"]),
                       "Rain100L (rain)", strip_x2=True)
            break

# WTT
if DATA_LOCAL and os.path.isdir(DATA_LOCAL):
    for split in ["test","val"]:
        sp = os.path.join(DATA_LOCAL, split)
        if os.path.isdir(sp):
            _add_pairs(os.path.join(sp,"images"), os.path.join(sp,"clean_images"),
                       f"WTT Synthetic ({split})")
            break

print(f"Dataset pool: {len(DATASET_POOL)} images available")
for lbl in sorted(set(t[2] for t in DATASET_POOL)):
    n = sum(1 for t in DATASET_POOL if t[2]==lbl)
    print(f"  {n:4d}  {lbl}")

# ── Pick image ────────────────────────────────────────────────
if TEST_IMAGE is not None:
    # User-supplied path
    assert os.path.exists(TEST_IMAGE), f"Not found: {TEST_IMAGE}"
    inp_path = TEST_IMAGE
    gt_path  = None
    src_label = f"Custom: {os.path.basename(TEST_IMAGE)}"
    print(f"\nUsing: {TEST_IMAGE}")
else:
    # Random from pool, optionally filtered by DEGRADATION keyword
    pool = DATASET_POOL
    if DEGRADATION:
        kw = DEGRADATION.lower()
        pool = [t for t in pool if kw in t[2].lower()]
        if not pool:
            print(f"No images for '{DEGRADATION}' — using full pool")
            pool = DATASET_POOL
    inp_path, gt_path, src_label = random.choice(pool)
    print(f"\nRandom pick: {os.path.basename(inp_path)}  [{src_label}]")

# ── Enhance ───────────────────────────────────────────────────
def _load_np(p):
    return np.array(Image.open(p).convert("RGB")).astype(np.float32)/255.0

def _enhance(pil_img, model):
    IW, IH = pil_img.size
    PW, PH = ((IW+3)//4)*4, ((IH+3)//4)*4
    pad = Image.new("RGB",(PW,PH)); pad.paste(pil_img,(0,0))
    t = transforms.ToTensor()(pad).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(t)
    enh = out["enhanced"][0].cpu().numpy().transpose(1,2,0)[:IH,:IW]
    fw  = out["filter_weights"][0].cpu().numpy()
    H   = out["weight_entropy"][0].item()
    wl  = out["weather_logits"][0]
    tl  = out["time_logits"][0]
    il  = out["illum_logits"][0]
    return np.clip(enh,0,1), fw, H, wl, tl, il

inp_pil  = Image.open(inp_path).convert("RGB")
inp_np   = _load_np(inp_path)
enh_np, fw, H, wl, tl, il = _enhance(inp_pil, model)

# Ground truth
has_gt  = (gt_path is not None and os.path.exists(gt_path))
gt_np   = _load_np(gt_path) if has_gt else None
if has_gt and gt_np.shape != enh_np.shape:
    IW2,IH2 = inp_pil.size
    gt_np = np.array(Image.open(gt_path).convert("RGB")
                     .resize((IW2,IH2),Image.LANCZOS)).astype(np.float32)/255.0

# Metrics
if has_gt:
    m_i = {"psnr":psnr_fn(gt_np,inp_np,data_range=1.0),
            "ssim":ssim_fn(gt_np,inp_np,data_range=1.0,channel_axis=2)}
    m_e = {"psnr":psnr_fn(gt_np,enh_np,data_range=1.0),
            "ssim":ssim_fn(gt_np,enh_np,data_range=1.0,channel_axis=2)}
    dp   = m_e["psnr"]-m_i["psnr"]
    ds   = m_e["ssim"]-m_i["ssim"]

# EPE predictions
WEATHER_NAMES = ["Clear","Rain","Fog","Light Snow","Glare"]
TIME_NAMES    = ["Dawn","Day","Dusk","Night"]
ILLUM_NAMES   = ["Low","Medium","High"]
import torch.nn.functional as TF2
pred_w = WEATHER_NAMES[wl.argmax().item()]
pred_t = TIME_NAMES[tl.argmax().item()]
pred_i = ILLUM_NAMES[il.argmax().item()]

# ── Figure: main comparison grid ──────────────────────────────
ncols  = 3 if has_gt else 2
fig    = plt.figure(figsize=(6*ncols, 5.5), facecolor="white")
gs     = GS.GridSpec(1, ncols, figure=fig, hspace=0.05, wspace=0.05)

# Panel 0 — Input
ax0 = fig.add_subplot(gs[0,0])
ax0.imshow(np.clip(inp_np,0,1)); ax0.axis("off")
t0  = f"Input  [{src_label}]"
if has_gt: t0 += f"\nPSNR={m_i['psnr']:.2f} dB   SSIM={m_i['ssim']:.4f}"
ax0.set_title(t0, fontsize=9, pad=6)

# Panel 1 — Enhanced
ax1 = fig.add_subplot(gs[0,1])
ax1.imshow(np.clip(enh_np,0,1)); ax1.axis("off")
col = "#1a7a1a" if (has_gt and dp>=0) else ("#cc2222" if has_gt else "black")
t1  = f"Enhanced  [EPE: {pred_w} · {pred_t} · {pred_i}]\nH={H:.3f}"
if has_gt: t1 += f"   PSNR={m_e['psnr']:.2f} dB   SSIM={m_e['ssim']:.4f}   ΔPSNR={dp:+.2f}"
ax1.set_title(t1, fontsize=9, pad=6, color=col)

# Panel 2 — GT
if has_gt:
    ax2 = fig.add_subplot(gs[0,2])
    ax2.imshow(np.clip(gt_np,0,1)); ax2.axis("off")
    ax2.set_title("Ground Truth", fontsize=9, pad=6)

fig.suptitle(f"EAIM-Net v5 — Full Pipeline Test", fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
p1 = os.path.join(DRIVE_RESULTS, "test_comparison.jpg")
fig.savefig(p1, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {p1}")

# ── Figure: filter weights bar chart ──────────────────────────
FILTER_COLORS = ["#2278CF","#1D9E75","#EF9F27","#7F77DD","#D85A30"]
fig2, ax = plt.subplots(figsize=(8,3), facecolor="white")
bars = ax.bar(FILTER_NAMES, fw, color=FILTER_COLORS, width=0.6, edgecolor="white")
ax.set_ylim(0, 1); ax.set_ylabel("Blending weight"); ax.set_title(
    f"ESS filter weights   H={H:.3f}   τ={model.ess.tau.item():.4f}", fontsize=10)
ax.axhline(0.2, color="gray", lw=0.8, ls="--", label="H>0.2 threshold")
ax.legend(fontsize=8)
for bar, v in zip(bars, fw):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f"{v:.3f}",
            ha="center", va="bottom", fontsize=8)
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout()
p2 = os.path.join(DRIVE_RESULTS, "test_filter_weights.jpg")
fig2.savefig(p2, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {p2}")

# ── Figure: difference map ─────────────────────────────────────
diff_amp = np.clip(np.abs(enh_np-inp_np)*5, 0, 1)
fig3, axes = plt.subplots(1,3,figsize=(15,4), facecolor="white")
axes[0].imshow(np.clip(inp_np,0,1));  axes[0].axis("off"); axes[0].set_title("Input",fontsize=9)
axes[1].imshow(np.clip(enh_np,0,1)); axes[1].axis("off"); axes[1].set_title("Enhanced",fontsize=9)
im = axes[2].imshow(diff_amp, cmap="hot"); axes[2].axis("off")
axes[2].set_title("Difference map (×5)\nBright = changed most", fontsize=9)
plt.colorbar(im, ax=axes[2], shrink=0.8)
fig3.suptitle("What the model changed", fontsize=10, fontweight="bold")
plt.tight_layout()
p3 = os.path.join(DRIVE_RESULTS, "test_diff_map.jpg")
fig3.savefig(p3, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {p3}")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*55)
print("  EAIM-Net v5 — Enhancement Result")
print("="*55)
print(f"  Image       : {os.path.basename(inp_path)}")
print(f"  Source      : {src_label}")
print(f"  EPE pred    : {pred_w} · {pred_t} · Illum={pred_i}")
print(f"  Entropy H   : {H:.4f}  ({'genuine blend' if H>0.2 else 'WARNING: low blending'})")
print(f"  Temperature : {model.ess.tau.item():.4f}")
print()
print("  Filter weights:")
for name, w_val in zip(FILTER_NAMES, fw):
    bar = "█" * int(w_val*30)
    print(f"  {name:<22} {bar:<30} {w_val:.4f}")
if has_gt:
    print()
    status = "IMPROVED ✓" if dp>0 else "REGRESSED ✗"
    print(f"  PSNR  : {m_i['psnr']:.2f} → {m_e['psnr']:.2f} dB  ({dp:+.2f})  {status}")
    print(f"  SSIM  : {m_i['ssim']:.4f} → {m_e['ssim']:.4f}  ({ds:+.4f})")
print("="*55)
print("\nRe-run cell for another random image.")
print("Set TEST_IMAGE = '/your/path.jpg' to test a specific image.")
print("Set DEGRADATION = 'rain' to pick from rain images only.")


---
## Cell 10 — Single Image Test

In [ ]:
# ================================================================
# CELL 10 — SINGLE IMAGE DEEP-DIVE
# Set DEG_PATH + optional GT_PATH. Shows 3 full figures.
# ================================================================
from complete_model import AdaptiveEnhancementModel

# ── EDIT THESE ───────────────────────────────────────────────
DEG_PATH  = "/content/weather_time_data/test/images/img1032_clear_night.jpg"
GT_PATH   = "/content/weather_time_data/test/clean_images/img1032_clear_night.jpg"
SAVE_FIGS = True
# ─────────────────────────────────────────────────────────────

# Load model
best=find_latest(DRIVE_CKPTS)
assert best, f"No checkpoint in {DRIVE_CKPTS}"
ck=torch.load(best,map_location=DEVICE)
INF=AdaptiveEnhancementModel(
    backbone=ck["config"]["model"]["backbone"],
    num_weather_classes=ck["config"]["model"]["num_weather_classes"],
    num_time_classes=ck["config"]["model"]["num_time_classes"],
    num_illum_classes=ck["config"]["model"]["num_illum_classes"],
    feature_dim=ck["config"]["model"]["feature_dim"],
    num_filters=ck["config"]["model"]["num_filters"],
    pretrained=False,
).to(DEVICE)
try:    INF.load_state_dict(ck["model_state_dict"])
except: INF.load_state_dict(ck["model_state_dict"],strict=False)
INF.eval()
print(f"Model: epoch={ck['epoch']}  version={ck.get('version','?')}")

def _pad4(pil):
    w,h=pil.size; pw,ph=((w+3)//4)*4,((h+3)//4)*4
    if pw==w and ph==h: return pil,(w,h)
    p=Image.new("RGB",(pw,ph),(0,0,0)); p.paste(pil,(0,0)); return p,(w,h)

@torch.no_grad()
def enhance(pil):
    pil_p,(ow,oh)=_pad4(pil)
    arr=np.array(pil_p).astype(np.float32)/255
    t=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).to(DEVICE)
    out=INF(t)
    fo=out["filter_outputs"][0].cpu().numpy()
    def _c(tensor): return np.clip(tensor[0].cpu().numpy().transpose(1,2,0)[:oh,:ow],0,1)
    return {"enhanced":_c(out["enhanced"]),
            "filter_imgs":[np.clip(fo[k].transpose(1,2,0)[:oh,:ow],0,1) for k in range(5)],
            "fw":out["filter_weights"][0].cpu().numpy(),
            "entropy":out["weight_entropy"][0].item(),
            "weather":WEATHER_NAMES[out["weather_logits"].argmax(1).item()],
            "time":   TIME_NAMES[out["time_logits"].argmax(1).item()],
            "illum":  ILLUM_NAMES[out["illum_logits"].argmax(1).item()],
            "tau":INF.ess.tau.item(),
            "w_probs":out["weather_logits"].softmax(1)[0].cpu().numpy(),
            "t_probs":out["time_logits"].softmax(1)[0].cpu().numpy(),
            "i_probs":out["illum_logits"].softmax(1)[0].cpu().numpy()}

deg_pil=Image.open(DEG_PATH).convert("RGB")
gt_pil =Image.open(GT_PATH).convert("RGB") if GT_PATH and os.path.exists(GT_PATH) else None
deg_np =np.array(deg_pil).astype(np.float32)/255
res=enhance(deg_pil)
gt_np =np.array(gt_pil.resize(deg_pil.size,Image.LANCZOS)).astype(np.float32)/255 if gt_pil else None
m_enh =metrics(res["enhanced"],gt_np) if gt_np is not None else None
m_deg =metrics(deg_np,gt_np)          if gt_np is not None else None
fw=res["fw"]; stem=os.path.splitext(os.path.basename(DEG_PATH))[0]

# FIG A: Comparison + weather probs
has_gt=gt_np is not None; nc=2+int(has_gt)
fig_a,axs=plt.subplots(1,nc+1,figsize=(5*(nc+1),5),gridspec_kw={"width_ratios":[3]*nc+[2.2]})
axs[0].imshow(deg_np); axs[0].axis("off")
t0="Degraded"
if m_deg: t0+=f"\nPSNR={m_deg['psnr']:.2f} dB  SSIM={m_deg['ssim']:.4f}"
axs[0].set_title(t0,fontsize=10)
imp=m_enh is not None and m_enh["psnr"]>m_deg["psnr"]
oc="#1a7a1a" if imp else ("#cc2222" if m_enh else "black")
axs[1].imshow(res["enhanced"]); axs[1].axis("off")
t1=f"Enhanced [{res['weather']} | {res['time']} | {res['illum']}]"
if m_enh:
    dp=m_enh["psnr"]-m_deg["psnr"]; ds=m_enh["ssim"]-m_deg["ssim"]
    t1+=f"\nPSNR={m_enh['psnr']:.2f} (d{dp:+.2f})  SSIM={m_enh['ssim']:.4f} (d{ds:+.4f})"
axs[1].set_title(t1,fontsize=10,color=oc)
if has_gt:
    axs[2].imshow(gt_np); axs[2].axis("off")
    t2="Ground Truth"
    if m_enh: t2+=f"\nLPIPS={m_enh['lpips']:.4f}"
    axs[2].set_title(t2,fontsize=10)
ax_wp=axs[-1]; probs=res["w_probs"]
cw=["#639922" if i==probs.argmax() else "#b0c4de" for i in range(5)]
bw=ax_wp.barh(WEATHER_NAMES,probs,color=cw,height=0.55,edgecolor="white")
ax_wp.set_xlim(0,1.1); ax_wp.set_xlabel("Probability",fontsize=9)
for bar,v in zip(bw,probs):
    ax_wp.text(v+0.01,bar.get_y()+bar.get_height()/2,f"{v:.3f}",va="center",fontsize=9,
               fontweight="bold" if v==probs.max() else "normal")
ax_wp.set_title(f"Weather probs\nt={res['tau']:.3f}  H={res['entropy']:.3f}",fontsize=9)
fig_a.suptitle(f"EAIM-Net v5 — {stem}",fontsize=12,fontweight="bold",y=1.02)
plt.tight_layout()
if SAVE_FIGS: fig_a.savefig(os.path.join(DRIVE_RESULTS,f"{stem}_comparison.jpg"),dpi=130,bbox_inches="tight"); print(f"Saved comparison")
plt.show()

# FIG B: Filter outputs + diff maps
fig_b,ab=plt.subplots(2,6,figsize=(23,8))
ab[0,0].imshow(deg_np);          ab[0,0].axis("off"); ab[0,0].set_title("Input",fontsize=9,fontweight="bold")
ab[1,0].imshow(res["enhanced"]); ab[1,0].axis("off"); ab[1,0].set_title("Enhanced",fontsize=9,fontweight="bold")
for k in range(5):
    v_=fw[k]; act=v_>0.10
    ab[0,k+1].imshow(res["filter_imgs"][k]); ab[0,k+1].axis("off")
    ab[0,k+1].set_title(f"{FILTER_NAMES[k]}\nw={v_:.4f}",fontsize=8,
                          color=FILTER_COLORS[k] if act else "#999",fontweight="bold" if act else "normal")
    if act:
        for sp_ in ab[0,k+1].spines.values():
            sp_.set_visible(True); sp_.set_edgecolor(FILTER_COLORS[k]); sp_.set_linewidth(3)
    diff=(res["filter_imgs"][k].astype(np.float64)-deg_np)*4+0.5
    rmse=float(np.sqrt(np.mean((res["filter_imgs"][k]-deg_np)**2)))
    ab[1,k+1].imshow(np.clip(diff,0,1)); ab[1,k+1].axis("off")
    ab[1,k+1].set_title(f"Delta x4  filter {k+1}\nRMSE={rmse:.4f}",fontsize=7,color="#555")
fig_b.suptitle("Filter outputs (top, green border=active w>0.10)  |  Diff maps x4 (bottom)",fontsize=10,fontweight="bold",y=1.01)
plt.tight_layout()
if SAVE_FIGS: fig_b.savefig(os.path.join(DRIVE_RESULTS,f"{stem}_filters.jpg"),dpi=130,bbox_inches="tight")
plt.show()

# FIG C: Weight bar + time/illum probs + metric card
fig_c=plt.figure(figsize=(16,5))
gs_c=GS.GridSpec(1,3,figure=fig_c,wspace=0.38)
a_fw=fig_c.add_subplot(gs_c[0]); a_ti=fig_c.add_subplot(gs_c[1]); a_tx=fig_c.add_subplot(gs_c[2])
bfw=a_fw.barh(FILTER_NAMES,fw,color=FILTER_COLORS,height=0.55,edgecolor="white")
a_fw.set_xlim(0,1.12); a_fw.set_xlabel("Weight",fontsize=10)
a_fw.axvline(0.10,color="gray",ls=":",lw=1,label="active (>0.10)"); a_fw.legend(fontsize=8)
a_fw.set_title("ESS Filter Weights",fontsize=11,fontweight="bold")
for bar,v in zip(bfw,fw):
    a_fw.text(v+0.01,bar.get_y()+bar.get_height()/2,f"{v:.4f}",va="center",fontsize=9,fontweight="bold" if v>0.10 else "normal")
x4=np.arange(4)
a_ti.bar(x4-0.18,res["t_probs"],0.35,label="Time", color="#5B8DB8",edgecolor="white")
a_ti.bar(x4[:3]+0.18,res["i_probs"],0.35,label="Illum",color="#E8A838",edgecolor="white")
a_ti.set_xticks(x4); a_ti.set_xticklabels(TIME_NAMES,fontsize=9)
a_ti.set_ylim(0,1.15); a_ti.set_ylabel("Probability",fontsize=9)
a_ti.legend(fontsize=9); a_ti.set_title("Time & Illum Probs",fontsize=11,fontweight="bold")
a_tx.axis("off")
n_act=sum(1 for v in fw if v>0.10)
txt=[f"  Weather : {res['weather']}",f"  Time    : {res['time']}",f"  Illum   : {res['illum']}",
     f"  Tau t   : {res['tau']:.4f}",f"  Entropy : {res['entropy']:.4f}",f"  Active  : {n_act}/5",""]
if m_enh:
    dp=m_enh["psnr"]-m_deg["psnr"]; ds=m_enh["ssim"]-m_deg["ssim"]
    txt+=[f"  PSNR  : {m_enh['psnr']:.4f} dB",f"  dPSNR : {dp:>+.4f} dB",
          f"  SSIM  : {m_enh['ssim']:.4f}",f"  dSSIM : {ds:>+.4f}",
          f"  LPIPS : {m_enh['lpips']:.4f}","",f"  Status: {'IMPROVED' if dp>0 else 'REGRESSED'}"]
else: txt.append("  No GT provided")
a_tx.text(0.04,0.97,"\n".join(txt),transform=a_tx.transAxes,fontsize=9,va="top",fontfamily="monospace",
          bbox=dict(boxstyle="round,pad=0.6",fc="#f7f7f7",ec="#ccc",lw=1.5))
fig_c.suptitle(f"Analysis: {stem}",fontsize=12,fontweight="bold")
if SAVE_FIGS: fig_c.savefig(os.path.join(DRIVE_RESULTS,f"{stem}_analysis.jpg"),dpi=130,bbox_inches="tight")
plt.show()

print("-"*52); print(f"  {stem}")
print(f"  Weather: {res['weather']} | {res['time']} | {res['illum']}")
print(f"  Tau={res['tau']:.3f}  H={res['entropy']:.3f}  Active={n_act}/5")
if m_enh:
    print(f"  PSNR  : {m_enh['psnr']:.4f} dB  (d{m_enh['psnr']-m_deg['psnr']:+.4f})")
    print(f"  SSIM  : {m_enh['ssim']:.4f}    (d{m_enh['ssim']-m_deg['ssim']:+.4f})")
    print(f"  LPIPS : {m_enh['lpips']:.4f}")
print("-"*52)


---
## Cell 11 — Full Test-Set Evaluation

In [ ]:
# ================================================================
# CELL 11 — FULL EVALUATION
# PSNR/SSIM/LPIPS on all test datasets
# Shows: table, per-weather breakdown, heatmap, gallery
# ================================================================
from dataset import EnvironmentalTextDataset, RealPairDataset, safe_collate

EVAL_BSZ=8; N_GALLERY=12

def _cnt(r):
    if not r or not os.path.isdir(r): return 0
    n=0
    for _,_,fs in os.walk(r): n+=sum(1 for f in fs if f.lower().endswith((".jpg",".jpeg",".png")))
    return n

def _make_loader(root,split,dtype):
    if _cnt(root)==0: return None
    sd=os.path.join(root,split)
    if not os.path.isdir(sd): sd=root
    if dtype=="synthetic":
        ds=EnvironmentalTextDataset(data_root=sd,split="test",image_size=256,paired=True,augment=False)
    else:
        ds=RealPairDataset(root=root,split=split,image_size=256,augment=False,dataset_type=dtype,debug=False)
    if len(ds)==0: return None
    return DataLoader(ds,batch_size=EVAL_BSZ,shuffle=False,num_workers=2,pin_memory=True,collate_fn=safe_collate)

ALL_REC=[]; ALL_ST={}
eval_sets=[
    ("Weather_Time_Text",DATA_LOCAL,   "test","synthetic"),
    ("LOL",              LOL_ROOT,     "test","lol"),
    ("Rain100L",         RAIN100L_ROOT,"test","rain100l"),
    ("Rain100H",         RAIN100H_ROOT,"test","rain100h"),
]
for ds_lbl,root,split,dtype in eval_sets:
    loader=_make_loader(root,split,dtype)
    if loader is None: print(f"  skipped: {ds_lbl}"); continue
    print(f"  Evaluating {ds_lbl}  ({len(loader)} batches)")
    ps,ss,ls=[],[],[]
    byw={i:{"ps":[],"ss":[],"ls":[],"dp":[]} for i in range(5)}
    rd=[]
    INF.eval()
    with torch.no_grad():
        for batch in tqdm(loader,desc=ds_lbl,leave=True):
            imgs=batch["image"].to(DEVICE); w_lbl=batch["weather_label"]
            if "clean_image" not in batch: continue
            clean=batch["clean_image"].to(DEVICE)
            out=INF(imgs); enh=out["enhanced"]
            import torch.nn.functional as TF2
            if enh.shape!=clean.shape:
                enh=TF2.interpolate(enh,size=clean.shape[2:],mode="bilinear",align_corners=False)
            fw_b=out["filter_weights"].cpu().numpy()
            ent_b=out["weight_entropy"].cpu().numpy()
            wp=out["weather_logits"].argmax(1).cpu().numpy()
            pnp=enh.cpu().numpy().transpose(0,2,3,1)
            gnp=clean.cpu().numpy().transpose(0,2,3,1)
            dnp=imgs.cpu().numpy().transpose(0,2,3,1)
            for i in range(imgs.shape[0]):
                me=metrics(pnp[i],gnp[i]); md_=metrics(dnp[i],gnp[i])
                wc=min(w_lbl[i].item(),4)
                ps.append(me["psnr"]); ss.append(me["ssim"]); ls.append(me["lpips"])
                byw[wc]["ps"].append(me["psnr"]); byw[wc]["ss"].append(me["ssim"])
                byw[wc]["ls"].append(me["lpips"]); byw[wc]["dp"].append(me["psnr"]-md_["psnr"])
                rd.append({"dataset":ds_lbl,"weather":wc,"deg":dnp[i].copy(),
                           "enh":pnp[i].copy(),"gt":gnp[i].copy(),
                           "psnr_in":md_["psnr"],"psnr_out":me["psnr"],
                           "ssim_in":md_["ssim"],"ssim_out":me["ssim"],
                           "lpips":me["lpips"],"dpsnr":me["psnr"]-md_["psnr"],
                           "dssim":me["ssim"]-md_["ssim"],
                           "fw":fw_b[i].copy(),"entropy":float(ent_b[i]),
                           "name":batch["image_name"][i] if "image_name" in batch else str(i)})
    if ps:
        ALL_ST[ds_lbl]={"n":len(ps),"psnr":float(np.mean(ps)),"ssim":float(np.mean(ss)),"lpips":float(np.mean(ls)),
                        "by_weather":{WEATHER_NAMES[k]:{"n":len(v["ps"]),
                            "psnr":float(np.mean(v["ps"])) if v["ps"] else float("nan"),
                            "ssim":float(np.mean(v["ss"])) if v["ss"] else float("nan"),
                            "lpips":float(np.mean(v["ls"]))if v["ls"] else float("nan"),
                            "dpsnr":float(np.mean(v["dp"]))if v["dp"] else float("nan"),
                        } for k,v in byw.items() if v["ps"]}}
        ALL_REC.extend(rd)

# Print table
print("\n"+"="*72)
print(f"  EAIM-Net v5 — Results  checkpoint: {os.path.basename(best)}")
print("="*72)
print(f"  {'Dataset':<24} {'N':>5}  {'PSNR':>8}  {'SSIM':>8}  {'LPIPS':>8}")
print("  "+"-"*60)
for lbl,r in ALL_ST.items():
    print(f"  {lbl:<24} {r['n']:>5}  {r['psnr']:>8.4f}  {r['ssim']:>8.4f}  {r['lpips']:>8.4f}")
print("="*72)

for lbl,r in ALL_ST.items():
    if not r.get("by_weather"): continue
    print(f"\n  {lbl} per weather:")
    print(f"    {'Condition':<14} {'N':>5}  {'PSNR':>8}  {'SSIM':>8}  {'DPSNR':>7}")
    for wn,wm in r["by_weather"].items():
        if math.isnan(wm["psnr"]): continue
        print(f"    {'OK' if wm['dpsnr']>0 else 'WRN'} {wn:<12} {wm['n']:>5}  {wm['psnr']:>8.4f}  {wm['ssim']:>8.4f}  {wm['dpsnr']:>+7.4f}")

with open(os.path.join(DRIVE_RESULTS,"metrics.json"),"w") as jf:
    def _jfy(o):
        if isinstance(o,dict): return {k:_jfy(v) for k,v in o.items()}
        if isinstance(o,list): return [_jfy(i) for i in o]
        if isinstance(o,float) and math.isnan(o): return None
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.integer,)):  return int(o)
        return o
    json.dump(_jfy(ALL_ST),jf,indent=2)
print(f"  JSON saved: {DRIVE_RESULTS}/metrics.json")

if ALL_REC:
    # Heatmap
    by_wn={}
    for rec in ALL_REC:
        wn=WEATHER_NAMES[rec["weather"]] if rec["weather"]<5 else "?"
        if wn not in by_wn: by_wn[wn]={"fw":[],"ent":[]}
        by_wn[wn]["fw"].append(rec["fw"]); by_wn[wn]["ent"].append(rec["entropy"])
    rows=list(by_wn.keys())
    matrix=np.array([np.mean(by_wn[w]["fw"],axis=0) for w in rows])
    ent_r=[np.mean(by_wn[w]["ent"]) for w in rows]
    fig_h,(ax_hm,ax_en)=plt.subplots(1,2,figsize=(14,max(4,len(rows)+2)))
    im=ax_hm.imshow(matrix,cmap="YlOrRd",aspect="auto",vmin=0,vmax=1)
    ax_hm.set_xticks(range(5)); ax_hm.set_xticklabels(FILTER_NAMES,rotation=25,ha="right",fontsize=9)
    ax_hm.set_yticks(range(len(rows))); ax_hm.set_yticklabels(rows,fontsize=11)
    ax_hm.set_title("Mean Filter Weights per Condition",fontsize=10,fontweight="bold")
    for i in range(len(rows)):
        for j in range(5):
            v_=matrix[i,j]
            ax_hm.text(j,i,f"{v_:.2f}",ha="center",va="center",fontsize=11,
                       fontweight="bold",color="white" if v_>0.55 else "black")
    plt.colorbar(im,ax=ax_hm,shrink=0.8)
    ce=["#639922" if e>0.20 else "#cc4422" for e in ent_r]
    be=ax_en.barh(rows,ent_r,color=ce,height=0.5)
    ax_en.axvline(0.20,color="green",ls="--",lw=1.5,label="target H>0.20")
    ax_en.axvline(0.05,color="red",  ls="--",lw=1,  label="collapse")
    ax_en.set_xlim(0,1); ax_en.set_xlabel("Entropy H",fontsize=10)
    ax_en.set_title("Blending per Condition",fontsize=10,fontweight="bold"); ax_en.legend(fontsize=9)
    for bar_,e_ in zip(be,ent_r):
        ax_en.text(e_+0.01,bar_.get_y()+bar_.get_height()/2,f"{e_:.3f}",va="center",fontsize=10)
    plt.tight_layout()
    fig_h.savefig(os.path.join(DRIVE_RESULTS,"eval_heatmap.jpg"),dpi=130,bbox_inches="tight")
    plt.show()

    # Gallery
    samp=random.sample(ALL_REC,min(N_GALLERY,len(ALL_REC)))
    GCOLS=3; GROWS=math.ceil(len(samp)/GCOLS)
    fig_g=plt.figure(figsize=(GCOLS*12,GROWS*4.2))
    fig_g.suptitle(f"Evaluation Gallery  epoch={ck['epoch']}",fontsize=12,fontweight="bold",y=1.01)
    for idx,rec in enumerate(samp):
        r_=idx//GCOLS; c_=idx%GCOLS
        gs_=GS.GridSpec(GROWS,GCOLS*5,figure=fig_g,hspace=0.5,wspace=0.05); bx_=c_*5
        a0=fig_g.add_subplot(gs_[r_,bx_:bx_+1]); a1=fig_g.add_subplot(gs_[r_,bx_+1:bx_+3])
        a2=fig_g.add_subplot(gs_[r_,bx_+3:bx_+4]); a3=fig_g.add_subplot(gs_[r_,bx_+4:bx_+5])
        a0.imshow(rec["deg"]); a0.axis("off"); a0.set_title(f"Deg\nPSNR={rec['psnr_in']:.1f}",fontsize=7)
        ok_=rec["dpsnr"]>=0; wn_=WEATHER_NAMES[rec["weather"]] if rec["weather"]<5 else "?"
        a1.imshow(rec["enh"]); a1.axis("off")
        a1.set_title(f"{wn_}\nPSNR={rec['psnr_out']:.1f} (d{rec['dpsnr']:+.1f})",fontsize=7,color="#1a7a1a" if ok_ else "#cc2222")
        a2.imshow(rec["gt"]); a2.axis("off"); a2.set_title(f"GT\nSSIM={rec['ssim_out']:.3f}\nLPIPS={rec['lpips']:.3f}",fontsize=7)
        fw_=rec["fw"]
        a3.barh(range(5),fw_,color=FILTER_COLORS,height=0.6); a3.set_xlim(0,1); a3.set_yticks(range(5))
        a3.set_yticklabels([n.split()[0] for n in FILTER_NAMES],fontsize=5); a3.tick_params(axis="x",labelsize=5)
        a3.set_title(f"H={rec['entropy']:.2f}",fontsize=6)
    fig_g.savefig(os.path.join(DRIVE_RESULTS,"eval_gallery.jpg"),dpi=130,bbox_inches="tight"); plt.show()
    print(f"\nAll figures saved to {DRIVE_RESULTS}")


---
## Cell 12 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_LOGS}